<a href="https://colab.research.google.com/github/antoniovfonseca/compare-time-series/blob/main/notebooks/comparisonTimeSeries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Compare two time series of maps**

---

This notebook implements the analytical framework proposed by Pontius Jr. et al. to systematically compare two time series of raster maps, Time Series X and Time Series Y. The main goal is to transparently quantify and map the agreements and differences between the maps across space and time.

The analysis partitions the comparison into detailed components that answer specific geographic and temporal questions. The workflow is divided into three main methodological steps:

### 1. Presence Assessment
Evaluates the agreement concerning the presence of a category at each time point. The framework decomposes the comparison into exact components of agreement and difference:
*   **Hits:** Where both time series agree concerning the presence.
*   **False Alarms:** Where Time Series Y shows more presence than Time Series X at a specific time point.
*   **Misses:** Where Time Series X shows more presence than Time Series Y at a specific time point.
*   **Space Difference:** Where both time series agree on the total quantity of presence but disagree on its spatial location at a time point.
*   **Time Difference:** Where both time series agree on the location and quantity across the series but disagree on the specific time points.

### 2. Change Assessment
Analyzes the landscape dynamics by comparing the individual time intervals with the overall change during the temporal extent. This step evaluates landscape transformations by accounting for:
*   **Gross Gain and Gross Loss:** The magnitude of gross gains and gross losses.
*   **Net Change:** The net change, consolidating gains and losses.

### 3. Spatial Maps
In addition to the diverging stacked bar charts that summarize the affected areas, the notebook collapses the temporal dimension to generate pixel-by-pixel geographic visualizations:
*   **Hits Maps:** Maps the locations of consistent agreement between the time series throughout the time points.
*   **Difference Maps:** Reveals areas with systematic overestimation of the magnitude of presence or change, showing which time series shows more presence or more change.
*   **Temporal Allocation Maps:** Highlights the exact areas where the two time series agree on the total quantity of accumulated presence or change, but disagree exclusively on when these events occurred.

---
**Generated Outputs:** The notebook processes large raster data efficiently using windowed reading and exports analytical .csv tables, stacked bar charts, and .tif raster maps ready for ecological and spatial interpretation.



## **1.Enviroment Setup**

In [ ]:
# --- Installations ---
get_ipython().system('pip -q install matplotlib-scalebar matplotlib-map-utils')

# --- Standard Library Imports ---
import os
import re
from contextlib import ExitStack
from typing import List, Dict, Tuple, Union

# --- Third-party Data/Math Imports ---
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from tqdm.auto import tqdm
from pyproj import CRS, Geod, Transformer

# --- Matplotlib Imports ---
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize, LinearSegmentedColormap
from matplotlib.patches import Patch, Rectangle, FancyArrowPatch
from matplotlib_scalebar.scalebar import ScaleBar
from matplotlib_map_utils import north_arrow

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## **2.Input your data**

In [ ]:
# --- Variables ---
# Value indicating the absence of valid data in the geospatial raster maps.
# It ensures that invalid pixels are masked during the assessment.
no_data_value = 255

# Pixel value indicating the presence of the target class in the binary map.
presence_value = 1

# Pixel value indicating the absence of the target class in the binary map.
absence_value = 0

# File paths for Time Series X (e.g., the reference or baseline dataset).
time_series_x = [
    "/content/drive/MyDrive/Dataset/pie/pixel-based/extent/PIELTER_PB_2010.tif",
    "/content/drive/MyDrive/Dataset/pie/pixel-based/extent/PIELTER_PB_2012.tif",
    "/content/drive/MyDrive/Dataset/pie/pixel-based/extent/PIELTER_PB_2014.tif",
    "/content/drive/MyDrive/Dataset/pie/pixel-based/extent/PIELTER_PB_2016.tif",
    "/content/drive/MyDrive/Dataset/pie/pixel-based/extent/PIELTER_PB_2018.tif",
    "/content/drive/MyDrive/Dataset/pie/pixel-based/extent/PIELTER_PB_2021.tif"
]

# File paths for Time Series Y (e.g., the comparison or model dataset).
time_series_y = [
    "/content/drive/MyDrive/Dataset/pie/object-based/extent/PIELTER_OB_2010.tif",
    "/content/drive/MyDrive/Dataset/pie/object-based/extent/PIELTER_OB_2012.tif",
    "/content/drive/MyDrive/Dataset/pie/object-based/extent/PIELTER_OB_2014.tif",
    "/content/drive/MyDrive/Dataset/pie/object-based/extent/PIELTER_OB_2016.tif",
    "/content/drive/MyDrive/Dataset/pie/object-based/extent/PIELTER_OB_2018.tif",
    "/content/drive/MyDrive/Dataset/pie/object-based/extent/PIELTER_OB_2021.tif"
]


# Output path
output_path = "/content/drive/MyDrive/Assessments/compare-time-series/extent2"


## **3.Presence Assessment**

### 3.1 Compute Presence

In [ ]:
def compute_presence_metrics_block(
    array_x: np.ndarray,
    array_y: np.ndarray,
    no_data_val: int,
    presence_val: int
) -> Dict[str, int]:
    """
    Computes presence assessment metrics for a single spatial block based
    on Pontius Jr. et al. methodology using MINIMUM and MAXIMUM functions.

    Parameters
    ----------
    array_x : np.ndarray
        Data array from Time Series X.
    array_y : np.ndarray
        Data array from Time Series Y.
    no_data_val : int
        Pixel value indicating no data.
    presence_val : int
        Pixel value indicating presence.

    Returns
    -------
    Dict[str, int]
        Aggregated metrics (Hits, False Alarms, Misses,
        Sum X, Sum Y) for the current window.
    """
    # Create a mask for valid pixels (ignore no_data_value)
    valid_mask = (
        (array_x != no_data_val) &
        (array_y != no_data_val)
    )

    # Normalize arrays mathematically: 1 for presence, 0 for absence
    # Only valid pixels are considered
    x_val = np.where(
        (array_x == presence_val) & valid_mask,
        1,
        0
    )
    y_val = np.where(
        (array_y == presence_val) & valid_mask,
        1,
        0
    )

    # Eq 1 & 8: Hits (Ph) = MINIMUM(x, y)
    hits_arr = np.minimum(
        x_val,
        y_val
    )

    # Eq 2 & 11: False Alarms (Pf) = MAXIMUM(0, y - x)
    fa_arr = np.maximum(
        0,
        y_val - x_val
    )

    # Eq 3 & 12: Misses (Pm) = MAXIMUM(0, x - y)
    misses_arr = np.maximum(
        0,
        x_val - y_val
    )

    return {
        "hits_sum": int(np.sum(hits_arr)),
        "fa_sum": int(np.sum(fa_arr)),
        "misses_sum": int(np.sum(misses_arr)),
        "x_sum": int(np.sum(x_val)),
        "y_sum": int(np.sum(y_val))
    }


def calculate_presence_assessment(
    ts_x_paths: List[str],
    ts_y_paths: List[str],
    no_data_val: int = 255,
    presence_val: int = 1,
    output_csv: str = "presence_assessment_metrics.csv"
) -> pd.DataFrame:
    """
    Executes the overall Presence Assessment over geospatial time series.
    Processes large datasets via windowed reading and computes per-year
    and overall statistics, exporting them to a CSV.

    Parameters
    ----------
    ts_x_paths : List[str]
        Paths to Time Series X.
    ts_y_paths : List[str]
        Paths to Time Series Y.
    no_data_val : int, optional
        Value for invalid pixels. Default is 255.
    presence_val : int, optional
        Value for presence. Default is 1.
    output_csv : str, optional
        File path for the output CSV. Default is 'presence_assessment_metrics.csv'.

    Returns
    -------
    pd.DataFrame
        DataFrame containing metrics structured for a stacked bar chart.
    """
    results = []

    # Variables to accumulate overall metrics across all time points
    total_hits = 0
    total_sum_x = 0
    total_sum_y = 0
    total_space_diff = 0

    for idx, (x_path, y_path) in enumerate(
        tqdm(
            zip(ts_x_paths, ts_y_paths),
            total=len(ts_x_paths),
            desc="Presence Assessment"
        )
    ):
        # Attempt to extract year from the filename via regex, fallback to index
        match = re.search(
            r'\d{4}',
            x_path
        )
        year_label = match.group(0) if match else f"Time_{idx+1}"

        # Year-specific accumulators
        yr_hits = 0
        yr_sum_x = 0
        yr_sum_y = 0

        with rasterio.open(x_path) as src_x, rasterio.open(y_path) as src_y:
            # Process efficiently window by window
            for _, window in src_x.block_windows():
                arr_x = src_x.read(
                    1,
                    window=window
                )
                arr_y = src_y.read(
                    1,
                    window=window
                )

                metrics = compute_presence_metrics_block(
                    arr_x,
                    arr_y,
                    no_data_val,
                    presence_val
                )

                yr_hits += metrics["hits_sum"]
                yr_sum_x += metrics["x_sum"]
                yr_sum_y += metrics["y_sum"]

        # Eq 5: Space Difference (Pu) at time t
        # MINIMUM(SUM(x), SUM(y)) - Hits
        yr_space_diff = min(
            yr_sum_x,
            yr_sum_y
        ) - yr_hits

        # Eq 6 & 7: False Alarms and Misses evaluated for the whole year
        yr_fa = max(0, yr_sum_y - yr_sum_x)
        yr_misses = max(0, yr_sum_x - yr_sum_y)

        # Time Difference is evaluated properly over the entire series,
        # so we leave it as 0 for individual time points.
        results.append({
            "Year": year_label,
            "Hits": yr_hits,
            "False Alarms": yr_fa,
            "Misses": yr_misses,
            "Space Difference": yr_space_diff,
            "Time Difference": 0
        })

        # Accumulate totals
        total_hits += yr_hits
        total_sum_x += yr_sum_x
        total_sum_y += yr_sum_y
        total_space_diff += yr_space_diff

    # Eq 10: Time Difference (Pv) overall
    # MINIMUM(SUM_TOTAL(x), SUM_TOTAL(y)) - Total_Hits - Total_Space_Difference
    overall_time_diff = min(
        total_sum_x,
        total_sum_y
    ) - total_hits - total_space_diff

    # Eqs 11 & 12: Total False Alarms and Misses evaluated over the entire extent
    total_fa = max(0, total_sum_y - total_sum_x)
    total_misses = max(0, total_sum_x - total_sum_y)

    # Add the Total row for the stacked bar chart consistency
    results.append({
        "Year": "Total",
        "Hits": total_hits,
        "False Alarms": total_fa,
        "Misses": total_misses,
        "Space Difference": total_space_diff,
        "Time Difference": overall_time_diff
    })

    # Generate DataFrame and export to CSV
    df_metrics = pd.DataFrame(results)
    df_metrics.to_csv(
        output_csv,
        index=False
    )

    return df_metrics

# --- Execute Presence Assessment ---
# Define the output directory specifically for tables
tables_dir = os.path.join(
    output_path,
    "tables"
)

# Ensure the 'tables' directory exists
os.makedirs(
    tables_dir,
    exist_ok=True
)

# Define the full path for the output CSV file
csv_file_path = os.path.join(
    tables_dir,
    "presence_assessment_metrics.csv"
)

print(f"Executing Presence Assessment...\nSaving output to: {csv_file_path}\n")

# Execute the function using variables defined in the initial setup
df_presence_metrics = calculate_presence_assessment(
    ts_x_paths=time_series_x,
    ts_y_paths=time_series_y,
    no_data_val=no_data_value,
    presence_val=presence_value,
    output_csv=csv_file_path
)

# Display the resulting DataFrame
display(df_presence_metrics)


### 3.2 Plot Presence for each time point

In [ ]:
def plot_presence_assessment(csv_path: str, output_dir: str) -> None:
    """
    Reads the presence assessment metrics CSV, filters out the 'Total' row
    and 'Time Difference' column, and generates a stacked bar chart.
    Saves the output to a 'charts' directory in high resolution.

    Parameters
    ----------
    csv_path : str
        Path to the CSV file containing the presence assessment metrics.
    output_dir : str
        Directory where the 'charts' folder will be created and the
        resulting plot will be saved.

    Returns
    -------
    None
    """
    # Read the CSV file generated in the previous step
    df = pd.read_csv(csv_path)

    # 1. Prepare Data
    # Filter out the 'Total' row as we only want individual time points
    df_filtered = df[df['Year'] != 'Total'].copy()

    # Drop 'Time Difference' column because it is an aggregated metric
    if 'Time Difference' in df_filtered.columns:
        df_filtered = df_filtered.drop(columns=['Time Difference'])

    # Rename columns to match requested labels
    df_filtered = df_filtered.rename(columns={
        'Hits': 'Agreement',
        'False Alarms': 'Object-based > Pixel-based', # Time Series X > Time Series Y
        'Misses': 'Pixel-based > Object-based'        # Time Series Y > Time Series X
    })

    # Set 'Year' as the index to align the X-axis properly
    df_plot = df_filtered.set_index('Year')

    # Define the specific order of the stack from bottom to top in the chart
    metrics_order = [
        'Agreement',
        'Space Difference',
        'Object-based > Pixel-based',
        'Pixel-based > Object-based'
    ]

    # Dynamic Scaling Logic
    # Find the maximum total presence (sum of stacked components for the highest year)
    max_val = df_plot[metrics_order].sum(axis=1).max()

    if max_val >= 1_000_000_000_000:
        scale_factor = 1_000_000_000_000
        y_label = "Presence (trillion pixels)"
    elif max_val >= 1_000_000_000:
        scale_factor = 1_000_000_000
        y_label = "Presence (billion pixels)"
    elif max_val >= 1_000_000:
        scale_factor = 1_000_000
        y_label = "Presence (million pixels)"
    elif max_val >= 1_000:
        scale_factor = 1_000
        y_label = "Presence (thousand pixels)"
    elif max_val >= 100:
        scale_factor = 100
        y_label = "Presence (hundred pixels)"
    else:
        scale_factor = 1
        y_label = "Presence (pixels)"

    # Scale the data
    df_plot[metrics_order] = df_plot[metrics_order] / scale_factor

    # 2. Styling and Colors
    # Accessible color palette
    colors = [
        '#000000',  # Agreement
        '#228B22',  # Space Difference
        '#FF8C00',  # Time Series Y > Time Series X (False Alarms)
        '#8B4513'   # Time Series X > Time Series Y (Misses)
    ]

    # Generate Chart
    # FIXED FIGURE SIZE AND MARGINS
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.subplots_adjust(
        left=0.1,
        right=0.65,
        top=0.90,
        bottom=0.15
    )

    df_plot[metrics_order].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=colors,
        edgecolor='none',
        linewidth=0,
        width=0.85  # Increased width to make bars thicker and reduce spacing
    )

    # 3. Add titles and labels
    ax.set_title(
        'Presence Assessment Components per Time Point',
        fontsize=14,
        pad=10
    )
    ax.set_xlabel(
        'Time Point',
        fontsize=14
    )
    ax.set_ylabel(
        y_label,
        fontsize=14
    )

    # Customizing the Legend
    handles, labels = ax.get_legend_handles_labels()

    # User requested order for legend (Top to Bottom)
    desired_order = [
        'Pixel-based > Object-based',
        'Object-based > Pixel-based',
        'Space Difference',
        'Agreement'
    ]
    order_dict = {label: idx for idx, label in enumerate(labels)}

    ordered_handles = [handles[order_dict[lbl]] for lbl in desired_order]
    ordered_labels = [labels[order_dict[lbl]] for lbl in desired_order]

    # Place legend in the middle vertically to the right of the plot
    # Adjusted bbox_to_anchor x value from 1.05 to 1.02 to move legend closer
    # Added fontsize parameter to increase legend text size
    leg = ax.legend(
        ordered_handles,
        ordered_labels,
        bbox_to_anchor=(1.02, 0.5),
        loc='center left',
        frameon=False,
        fontsize=12
    )

    # Remove the border (frame) from the legend symbols
    for patch in leg.get_patches():
        patch.set_linewidth(0)
        patch.set_edgecolor('none')

    # Increase the size of the x and y axis ticks
    ax.tick_params(
        axis='both',
        which='major',
        labelsize=14
    )

    # Keep X-axis labels horizontal
    plt.xticks(rotation=0)

    # Remove tight_layout as it conflicts with subplots_adjust
    # plt.tight_layout()

    # 4. Save the figure in high resolution
    charts_dir = os.path.join(
        output_dir,
        "charts"
    )
    os.makedirs(
        charts_dir,
        exist_ok=True
    )
    output_file = os.path.join(
        charts_dir,
        "presence_assessment_chart.png"
    )

    # Remove bbox_inches='tight' to preserve exact figure size
    plt.savefig(
        output_file,
        dpi=300,
        format='png'
    )
    print(f"Chart successfully saved to: {output_file}\n")

    plt.show()

# Define the CSV path explicitly so the cell runs independently
tables_dir = os.path.join(
    output_path,
    "tables"
)
csv_file_path = os.path.join(
    tables_dir,
    "presence_assessment_metrics.csv"
)

# Execute the plotting function
plot_presence_assessment(csv_file_path, output_path)


### 3.3 Plot Sum chart

In [ ]:
def plot_sum_chart(csv_path: str, output_dir: str) -> None:
    """
    Reads the presence assessment metrics CSV, filters the 'Total' row,
    and generates a single stacked bar chart representing the sum over time.
    Saves the output to the 'charts' directory in high resolution.

    Parameters
    ----------
    csv_path : str
        Path to the CSV file containing the presence assessment metrics.
    output_dir : str
        Directory where the 'charts' folder will be created and the
        resulting plot will be saved.

    Returns
    -------
    None
    """
    # Read the CSV file
    df = pd.read_csv(csv_path)

    # 1. Prepare Data
    # Isolate the 'Total' row
    df_total = df[df['Year'] == 'Total'].copy()

    # Rename columns to match the previous chart
    df_total = df_total.rename(columns={
        'Hits': 'Agreement',
        'False Alarms': 'Object-based > Pixel-based',
        'Misses': 'Pixel-based > Object-based'
    })

    # Set 'Year' as index
    df_plot = df_total.set_index('Year')

    # Define the specific order of the stack from bottom to top
    metrics_order = [
        'Agreement',
        'Space Difference',
        'Time Difference',
        'Object-based > Pixel-based',
        'Pixel-based > Object-based'
    ]

    # Dynamic Scaling Logic
    max_val = df_plot[metrics_order].sum(axis=1).max()

    if max_val >= 1_000_000_000_000:
        scale_factor = 1_000_000_000_000
        y_label = "Presence (trillion pixels)"
    elif max_val >= 1_000_000_000:
        scale_factor = 1_000_000_000
        y_label = "Presence  (billion pixels)"
    elif max_val >= 1_000_000:
        scale_factor = 1_000_000
        y_label = "Presence  (million pixels)"
    elif max_val >= 1_000:
        scale_factor = 1_000
        y_label = "Presence  (thousand pixels)"
    elif max_val >= 100:
        scale_factor = 100
        y_label = "Presence (hundred pixels)"
    else:
        scale_factor = 1
        y_label = "Presence (Pixels)"

    # Scale the data
    df_plot[metrics_order] = df_plot[metrics_order] / scale_factor

    # 2. Styling and Colors
    colors = [
        '#000000',  # Agreement
        '#228B22',  # Space Difference
        '#800080',  # Time Difference
        '#FF8C00',  # Time Series Y > Time Series X
        '#8B4513'   # Time Series X > Time Series Y
    ]

    # Generate Chart
    # FIXED FIGURE SIZE AND MARGINS
    fig, ax = plt.subplots(figsize=(8, 6))
    fig.subplots_adjust(
        left=0.1,
        right=0.65,
        top=0.90,
        bottom=0.15
    )

    # Plot single stacked bar
    df_plot[metrics_order].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=colors,
        edgecolor='none',
        linewidth=0,
        width=1.0
    )

    # Reduce the x-axis limits to remove empty space on the sides of the bar
    ax.set_xlim(-0.6, 0.6)

    # 3. Add titles and labels
    ax.set_title(
        'Sum of Time Series',
        fontsize=14,
        pad=10
    )

    ax.set_ylabel(
        y_label,
        fontsize=14
    )

    # Set x-axis label to 'Sum' and ensure it is not rotated
    ax.set_xlabel('')
    ax.set_xticklabels(
        ['Sum'],
        fontsize=14,
        rotation=0
    )
    ax.tick_params(bottom=False)

    # Customizing the Legend (Top to Bottom)
    handles, labels = ax.get_legend_handles_labels()
    desired_order = [
        'Pixel-based > Object-based',
        'Object-based > Pixel-based',
        'Time Difference',
        'Space Difference',
        'Agreement'
    ]
    order_dict = {label: idx for idx, label in enumerate(labels)}
    ordered_handles = [handles[order_dict[lbl]] for lbl in desired_order]
    ordered_labels = [labels[order_dict[lbl]] for lbl in desired_order]

    leg = ax.legend(
        ordered_handles,
        ordered_labels,
        bbox_to_anchor=(1.02, 0.5),
        loc='center left',
        frameon=False,
        fontsize=12
    )

    for patch in leg.get_patches():
        patch.set_linewidth(0)
        patch.set_edgecolor('none')

    ax.tick_params(
        axis='y',
        which='major',
        labelsize=14
    )
    # Remove tight_layout
    # plt.tight_layout()

    # 4. Save the figure in high resolution
    charts_dir = os.path.join(
        output_dir,
        "charts"
    )
    os.makedirs(
        charts_dir,
        exist_ok=True
    )
    output_file = os.path.join(
        charts_dir,
        "presence_sum_chart.png"
    )

    # Remove bbox_inches='tight'
    plt.savefig(
        output_file,
        dpi=300,
        format='png'
    )
    print(f"Chart successfully saved to: {output_file}\n")
    plt.show()

# Define the CSV path explicitly so the cell runs independently
tables_dir = os.path.join(
    output_path,
    "tables"
)
csv_file_path = os.path.join(
    tables_dir,
    "presence_assessment_metrics.csv"
)

# Execute the function
plot_sum_chart(csv_file_path, output_path)


## **4.Change Assessment**

### 4.1 Compute Gross Change

In [ ]:
def compute_change(
    arr_x_t0: np.ndarray,
    arr_x_t1: np.ndarray,
    arr_y_t0: np.ndarray,
    arr_y_t1: np.ndarray,
    no_data_val: int,
    presence_val: int
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """
    Computes Gain and Loss metrics for a single spatial block.

    Parameters
    ----------
    arr_x_t0 : np.ndarray
        Array for Time Series X at time t0.
    arr_x_t1 : np.ndarray
        Array for Time Series X at time t1.
    arr_y_t0 : np.ndarray
        Array for Time Series Y at time t0.
    arr_y_t1 : np.ndarray
        Array for Time Series Y at time t1.
    no_data_val : int
        Pixel value indicating no data.
    presence_val : int
        Pixel value indicating presence.

    Returns
    -------
    Tuple[Dict[str, float], Dict[str, float]]
        A tuple containing two dictionaries, one for gain metrics and one for loss metrics.
    """
    # Create mask for valid pixels across all 4 arrays
    valid_mask = (
        (arr_x_t0 != no_data_val) &
        (arr_x_t1 != no_data_val) &
        (arr_y_t0 != no_data_val) &
        (arr_y_t1 != no_data_val)
    )

    # Normalize to 0 (absence) and 1 (presence)
    x0 = np.where(
        (arr_x_t0 == presence_val) & valid_mask,
        1,
        0
    )
    x1 = np.where(
        (arr_x_t1 == presence_val) & valid_mask,
        1,
        0
    )
    y0 = np.where(
        (arr_y_t0 == presence_val) & valid_mask,
        1,
        0
    )
    y1 = np.where(
        (arr_y_t1 == presence_val) & valid_mask,
        1,
        0
    )

    # Step 1: Interval Change Matrices
    gain_x = np.maximum(
        0,
        x1 - x0
    )
    gain_y = np.maximum(
        0,
        y1 - y0
    )
    loss_x = np.minimum(
        0,
        x1 - x0
    )
    loss_y = np.minimum(
        0,
        y1 - y0
    )

    # Step 2: Gain Metrics (Equations 1-12 adapted)
    gh = np.minimum(
        gain_x,
        gain_y
    )
    gf = np.maximum(
        0,
        gain_y - gain_x
    )
    gm = np.maximum(
        0,
        gain_x - gain_y
    )

    gains = {
        'hits': np.sum(gh),
        'fa': np.sum(gf),
        'misses': np.sum(gm),
        'sum_x': np.sum(gain_x),
        'sum_y': np.sum(gain_y)
    }

    # Step 3: Loss Metrics (Equations 17-28 adapted)
    lh = np.maximum(
        loss_x,
        loss_y
    )
    lf = np.minimum(
        0,
        loss_y - loss_x
    )
    lm = np.minimum(
        0,
        loss_x - loss_y
    )

    losses = {
        'hits': np.sum(lh),
        'fa': np.sum(lf),
        'misses': np.sum(lm),
        'sum_x': np.sum(loss_x),
        'sum_y': np.sum(loss_y)
    }

    return gains, losses


def calculate_change_assessment(
    ts_x_paths: List[str],
    ts_y_paths: List[str],
    no_data_val: int = 255,
    presence_val: int = 1,
    output_csv: str = "change_assessment_metrics.csv"
) -> pd.DataFrame:
    """
    Executes the Change Assessment over geospatial time series.

    Parameters
    ----------
    ts_x_paths : List[str]
        Paths to Time Series X.
    ts_y_paths : List[str]
        Paths to Time Series Y.
    no_data_val : int, optional
        Value for invalid pixels. Default is 255.
    presence_val : int, optional
        Value for presence. Default is 1.
    output_csv : str, optional
        File path for the output CSV. Default is 'change_assessment_metrics.csv'.

    Returns
    -------
    pd.DataFrame
        DataFrame containing change assessment metrics structured for plotting.
    """
    results = []

    # Accumulators for Total Sums
    total_gain = {
        'hits': 0,
        'fa': 0,
        'misses': 0,
        'sum_x': 0,
        'sum_y': 0,
        'space_diff': 0
    }
    total_loss = {
        'hits': 0,
        'fa': 0,
        'misses': 0,
        'sum_x': 0,
        'sum_y': 0,
        'space_diff': 0
    }

    # Process Intervals (t-1 to t)
    for t in tqdm(
        range(1, len(ts_x_paths)),
        desc="Change Assessment Intervals"
    ):
        path_x0 = ts_x_paths[t-1]
        path_x1 = ts_x_paths[t]
        path_y0 = ts_y_paths[t-1]
        path_y1 = ts_y_paths[t]

        m_x0 = re.search(
            r'\d{4}',
            path_x0
        )
        m_x1 = re.search(
            r'\d{4}',
            path_x1
        )
        yr0 = m_x0.group(0) if m_x0 else f"T{t-1}"
        yr1 = m_x1.group(0) if m_x1 else f"T{t}"
        interval_label = f"{yr0}-{yr1}"

        int_gain = {
            'hits': 0,
            'fa': 0,
            'misses': 0,
            'sum_x': 0,
            'sum_y': 0
        }
        int_loss = {
            'hits': 0,
            'fa': 0,
            'misses': 0,
            'sum_x': 0,
            'sum_y': 0
        }

        with rasterio.open(path_x0) as src_x0, \
             rasterio.open(path_x1) as src_x1, \
             rasterio.open(path_y0) as src_y0, \
             rasterio.open(path_y1) as src_y1:

            for _, window in src_x0.block_windows():
                arr_x0 = src_x0.read(
                    1,
                    window=window
                )
                arr_x1 = src_x1.read(
                    1,
                    window=window
                )
                arr_y0 = src_y0.read(
                    1,
                    window=window
                )
                arr_y1 = src_y1.read(
                    1,
                    window=window
                )

                gains, losses = compute_change(
                    arr_x0,
                    arr_x1,
                    arr_y0,
                    arr_y1,
                    no_data_val,
                    presence_val
                )

                for k in int_gain.keys():
                    int_gain[k] += gains[k]
                    int_loss[k] += losses[k]

        # Space Difference
        gu = min(
            int_gain['sum_x'],
            int_gain['sum_y']
        ) - int_gain['hits']
        lu = max(
            int_loss['sum_x'],
            int_loss['sum_y']
        ) - int_loss['hits']

        # Map-level Quantity Error (False Alarms & Misses) to avoid double counting
        gain_fa = max(0, int_gain['sum_y'] - int_gain['sum_x'])
        gain_misses = max(0, int_gain['sum_x'] - int_gain['sum_y'])
        loss_fa = max(0, abs(int_loss['sum_y']) - abs(int_loss['sum_x']))
        loss_misses = max(0, abs(int_loss['sum_x']) - abs(int_loss['sum_y']))

        results.append({
            'Time_Interval': interval_label,
            'Change_Type': 'Gain',
            'Hits': int_gain['hits'],
            'False Alarms': gain_fa,
            'Misses': gain_misses,
            'Space Difference': gu,
            'Time Difference': 0,
            'Alternation': 0
        })

        results.append({
            'Time_Interval': interval_label,
            'Change_Type': 'Loss',
            'Hits': abs(int_loss['hits']),
            'False Alarms': loss_fa,
            'Misses': loss_misses,
            'Space Difference': abs(lu),
            'Time Difference': 0,
            'Alternation': 0
        })

        # Accumulate totals
        for k in total_gain.keys():
            if k != 'space_diff':
                total_gain[k] += int_gain[k]
                total_loss[k] += int_loss[k]
        total_gain['space_diff'] += gu
        total_loss['space_diff'] += lu

    # Total Sum Metrics with Time Difference
    gv = min(
        total_gain['sum_x'],
        total_gain['sum_y']
    ) - total_gain['hits'] - total_gain['space_diff']

    lv = max(
        total_loss['sum_x'],
        total_loss['sum_y']
    ) - total_loss['hits'] - total_loss['space_diff']

    # Map-level Quantity Error (False Alarms & Misses) for Total Sum
    total_gain_fa = max(0, total_gain['sum_y'] - total_gain['sum_x'])
    total_gain_misses = max(0, total_gain['sum_x'] - total_gain['sum_y'])
    total_loss_fa = max(0, abs(total_loss['sum_y']) - abs(total_loss['sum_x']))
    total_loss_misses = max(0, abs(total_loss['sum_x']) - abs(total_loss['sum_y']))

    results.append({
        'Time_Interval': 'Total_Sum',
        'Change_Type': 'Gain',
        'Hits': total_gain['hits'],
        'False Alarms': total_gain_fa,
        'Misses': total_gain_misses,
        'Space Difference': total_gain['space_diff'],
        'Time Difference': gv,
        'Alternation': 0
    })
    results.append({
        'Time_Interval': 'Total_Sum',
        'Change_Type': 'Loss',
        'Hits': abs(total_loss['hits']),
        'False Alarms': total_loss_fa,
        'Misses': total_loss_misses,
        'Space Difference': abs(total_loss['space_diff']),
        'Time Difference': abs(lv),
        'Alternation': 0
    })

    # Temporal Extent (First to Last Map)
    ext_gain = {
        'hits': 0,
        'fa': 0,
        'misses': 0,
        'sum_x': 0,
        'sum_y': 0
    }
    ext_loss = {
        'hits': 0,
        'fa': 0,
        'misses': 0,
        'sum_x': 0,
        'sum_y': 0
    }

    with rasterio.open(ts_x_paths[0]) as src_x0, \
         rasterio.open(ts_x_paths[-1]) as src_x1, \
         rasterio.open(ts_y_paths[0]) as src_y0, \
         rasterio.open(ts_y_paths[-1]) as src_y1:

        for _, window in src_x0.block_windows():
            arr_x0 = src_x0.read(
                1,
                window=window
            )
            arr_x1 = src_x1.read(
                1,
                window=window
            )
            arr_y0 = src_y0.read(
                1,
                window=window
            )
            arr_y1 = src_y1.read(
                1,
                window=window
            )

            gains, losses = compute_change(
                arr_x0,
                arr_x1,
                arr_y0,
                arr_y1,
                no_data_val,
                presence_val
            )

            for k in ext_gain.keys():
                ext_gain[k] += gains[k]
                ext_loss[k] += losses[k]

    ext_gu = min(
        ext_gain['sum_x'],
        ext_gain['sum_y']
    ) - ext_gain['hits']

    ext_lu = max(
        ext_loss['sum_x'],
        ext_loss['sum_y']
    ) - ext_loss['hits']

    # Map-level Quantity Error (False Alarms & Misses) for Temporal Extent
    ext_gain_fa = max(0, ext_gain['sum_y'] - ext_gain['sum_x'])
    ext_gain_misses = max(0, ext_gain['sum_x'] - ext_gain['sum_y'])
    ext_loss_fa = max(0, abs(ext_loss['sum_y']) - abs(ext_loss['sum_x']))
    ext_loss_misses = max(0, abs(ext_loss['sum_x']) - abs(ext_loss['sum_y']))

    results.append({
        'Time_Interval': 'Temporal_Extent',
        'Change_Type': 'Gain',
        'Hits': ext_gain['hits'],
        'False Alarms': ext_gain_fa,
        'Misses': ext_gain_misses,
        'Space Difference': ext_gu,
        'Time Difference': 0,
        'Alternation': 0
    })
    results.append({
        'Time_Interval': 'Temporal_Extent',
        'Change_Type': 'Loss',
        'Hits': abs(ext_loss['hits']),
        'False Alarms': ext_loss_fa,
        'Misses': ext_loss_misses,
        'Space Difference': abs(ext_lu),
        'Time Difference': 0,
        'Alternation': 0
    })

    # Calculate Alternation for Total_Sum row (Absolute difference between Total and Extent)
    df = pd.DataFrame(results)

    # Update Alternation in Total_Sum
    metrics_cols = [
        'Hits',
        'False Alarms',
        'Misses',
        'Space Difference'
    ]
    for change_type in ['Gain', 'Loss']:
        mask_total = (
            (df['Time_Interval'] == 'Total_Sum') &
            (df['Change_Type'] == change_type)
        )
        mask_extent = (
            (df['Time_Interval'] == 'Temporal_Extent') &
            (df['Change_Type'] == change_type)
        )

        for col in metrics_cols:
            val_total = df.loc[mask_total, col].values[0]
            val_extent = df.loc[mask_extent, col].values[0]
            df.loc[mask_total, 'Alternation'] += abs(
                val_total - val_extent
            )

    df.to_csv(
        output_csv,
        index=False
    )
    print(f"CSV successfully saved to: {output_csv}\n")
    return df

# --- Execute Change Assessment ---
tables_dir = os.path.join(
    output_path,
    "tables"
)
os.makedirs(
    tables_dir,
    exist_ok=True
)
change_csv_path = os.path.join(
    tables_dir,
    "change_assessment_metrics.csv"
)

print(f"Executing Change Assessment...\nSaving output to: {change_csv_path}\n")

df_change_metrics = calculate_change_assessment(
    ts_x_paths=time_series_x,
    ts_y_paths=time_series_y,
    no_data_val=no_data_value,
    presence_val=presence_value,
    output_csv=change_csv_path
)

display(df_change_metrics)


#### 4.1.1 Plot Gross Change

In [ ]:
def plot_gross_change_chart(
    csv_path: str,
    output_dir: str
) -> None:
    """
    Reads the change assessment metrics CSV and generates a single diverging
    stacked bar chart for Gains (positive) and Losses (negative) per time interval.

    This function filters out aggregated metrics ('Total_Sum' and 'Temporal_Extent')
    and plots the gross change components using a custom blue color palette for
    Gains and a red color palette for Losses. The final chart is saved as a
    high-resolution PNG file.

    Parameters
    ----------
    csv_path : str
        Path to the CSV file containing the computed change assessment metrics.
    output_dir : str
        Directory where the 'charts' subfolder will be created and the
        resulting plot will be saved.

    Returns
    -------
    None
    """
    # 1. Prepare Data
    df = pd.read_csv(csv_path)

    # Filter out Total_Sum and Temporal_Extent
    df_filtered = df[
        ~df['Time_Interval'].isin(['Total_Sum', 'Temporal_Extent'])
    ].copy()

    # Drop columns not needed for this chart
    if 'Time Difference' in df_filtered.columns:
        df_filtered = df_filtered.drop(columns=['Time Difference'])
    if 'Alternation' in df_filtered.columns:
        df_filtered = df_filtered.drop(columns=['Alternation'])

    # Isolate metrics and invert Losses for the diverging chart
    metrics_order = [
        'Hits',
        'Space Difference',
        'False Alarms',
        'Misses'
    ]

    loss_mask = df_filtered['Change_Type'] == 'Loss'
    df_filtered.loc[loss_mask, metrics_order] *= -1

    # Separate into Gain and Loss for plotting on the same axis
    df_gain = df_filtered[
        df_filtered['Change_Type'] == 'Gain'
    ].set_index('Time_Interval')

    df_loss = df_filtered[
        df_filtered['Change_Type'] == 'Loss'
    ].set_index('Time_Interval')

    # Dynamic Scaling Logic
    max_val_gain = df_gain[metrics_order].sum(axis=1).max()
    max_val_loss = df_loss[metrics_order].abs().sum(axis=1).max()
    max_val = max(max_val_gain, max_val_loss)

    if max_val >= 1_000_000_000_000:
        scale_factor = 1_000_000_000_000
        y_label = "Gross Loss and Gross Gain (trillion pixels)"
    elif max_val >= 1_000_000_000:
        scale_factor = 1_000_000_000
        y_label = "Gross Loss and Gross Gain (billion pixels)"
    elif max_val >= 1_000_000:
        scale_factor = 1_000_000
        y_label = "Gross Loss and Gross Gain (million pixels)"
    elif max_val >= 1_000:
        scale_factor = 1_000
        y_label = "Gross Loss and Gross Gain (thousand pixels)"
    elif max_val >= 100:
        scale_factor = 100
        y_label = "Gross Loss and Gross Gain (hundred pixels)"
    else:
        scale_factor = 1
        y_label = "Gross Loss and Gross Gain (pixels)"

    # Scale the data
    df_gain[metrics_order] = df_gain[metrics_order] / scale_factor
    df_loss[metrics_order] = df_loss[metrics_order] / scale_factor
    max_val_scaled = max_val / scale_factor

    # 2. Styling and Colors
    # 4 shades of Blue for Gains (Dark to Light)
    colors_gain = [
        '#08306b',  # Gain Agreement (Hits)
        '#08519c',  # Gain Space Difference
        '#6baed6',  # Gain False Alarms
        '#bdd7e7'   # Gain Misses
    ]

    # 4 shades of Red for Losses (Dark to Light)
    colors_loss = [
        '#67000d',  # Loss Agreement (Hits)
        '#de2d26',  # Loss Space Difference
        '#fb6a4a',  # Loss Time Series Y > Time Series X (False Alarms)
        '#fcae91'   # Loss Time Series X > Time Series Y (Misses)
    ]

    # 3. Generate Chart (Single Axis)
    # FIXED FIGURE SIZE AND MARGINS
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.subplots_adjust(
        left=0.1,
        right=0.60,
        top=0.90,
        bottom=0.15
    )

    # Plot Gains (Positive)
    df_gain[metrics_order].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=colors_gain,
        edgecolor='none',
        linewidth=0,
        width=0.85,
        legend=False
    )

    # Plot Losses (Negative)
    df_loss[metrics_order].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=colors_loss,
        edgecolor='none',
        linewidth=0,
        width=0.85,
        legend=False
    )

    # Force symmetric Y-axis limits so Y=0 is exactly in the center
    y_limit = max_val_scaled * 1.05
    ax.set_ylim(-y_limit, y_limit)

    # Add a solid black line at Y=0 to clearly separate gains and losses
    ax.axhline(0, color='black', linewidth=1)

    # 4. Custom Legend
    legend_elements = [
        mpatches.Patch(
            color=colors_gain[3],
            label='Gain Pixel-based > Object-based'
        ),
        mpatches.Patch(
            color=colors_gain[2],
            label='Gain Object-based > Pixel-based'
        ),
        mpatches.Patch(
            color=colors_gain[1],
            label='Gain Space Difference'
        ),
        mpatches.Patch(
            color=colors_gain[0],
            label='Gain Agreement'
        ),
        mpatches.Patch(
            color=colors_loss[0],
            label='Loss Agreement'
        ),
        mpatches.Patch(
            color=colors_loss[1],
            label='Loss Space Difference'
        ),
        mpatches.Patch(
            color=colors_loss[2],
            label='Loss Object-based > Pixel-based'
        ),
        mpatches.Patch(
            color=colors_loss[3],
            label='Loss Pixel-based > Object-based'
        ),
    ]

    ax.legend(
        handles=legend_elements,
        bbox_to_anchor=(1.02, 0.5),
        loc='center left',
        frameon=False,
        fontsize=12
    )

    # 5. Styling and Formatting Clean
    ax.set_title(
        'Gross Gain and Loss During Time Interval',
        fontsize=14,
        pad=10
    )
    ax.set_ylabel(
        y_label,
        fontsize=14
    )

    # Force the X-axis label to be blank (overrides pandas default)
    ax.set_xlabel('')

    ax.tick_params(
        axis='both',
        which='major',
        labelsize=14
    )
    plt.xticks(rotation=45)

    # Disable grid lines for a clean background
    ax.grid(False)

    # Format Y-axis to show absolute values (remove minus signs)
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(lambda x, pos: f"{abs(x):g}")
    )

    # Save the figure
    charts_dir = os.path.join(
        output_dir,
        "charts"
    )
    os.makedirs(
        charts_dir,
        exist_ok=True
    )

    output_file = os.path.join(
        charts_dir,
        "gross_change_intervals_chart.png"
    )

    # Remove bbox_inches='tight' to preserve exact figure size
    plt.savefig(
        output_file,
        dpi=300,
        format='png'
    )
    print(f"Chart successfully saved to: {output_file}\n")

    plt.show()

# Define the CSV path explicitly so the cell runs independently
tables_dir = os.path.join(
    output_path,
    "tables"
)
change_csv_path = os.path.join(
    tables_dir,
    "change_assessment_metrics.csv"
)

# Execute the plotting function
plot_gross_change_chart(change_csv_path, output_path)


#### 4.1.2 Plot Gross Sum and Extent

In [ ]:
def plot_sum_and_extent_chart(
    csv_path: str,
    output_dir: str
) -> None:
    """
    Reads the change assessment metrics CSV and generates a single
    diverging stacked bar chart for 'Total_Sum' and 'Temporal_Extent'.

    Parameters
    ----------
    csv_path : str
        Path to the CSV file containing the computed change assessment metrics.
    output_dir : str
        Directory where the 'charts' subfolder will be created and the
        resulting plot will be saved.

    Returns
    -------
    None
    """
    # 1. Prepare Data
    df = pd.read_csv(csv_path)

    # Filter to keep ONLY 'Total_Sum' and 'Temporal_Extent'
    target_intervals = ['Total_Sum', 'Temporal_Extent']
    df_filtered = df[
        df['Time_Interval'].isin(target_intervals)
    ].copy()

    # Isolate the 5 specific metrics for stacking
    metrics_order = [
        'Hits',
        'Space Difference',
        'Time Difference',
        'False Alarms',
        'Misses'
    ]

    # Invert Losses to negative values for the diverging chart
    loss_mask = df_filtered['Change_Type'] == 'Loss'
    df_filtered.loc[loss_mask, metrics_order] *= -1

    # Separate into Gain and Loss for plotting on the same axis
    df_gain = df_filtered[
        df_filtered['Change_Type'] == 'Gain'
    ].set_index('Time_Interval')

    df_loss = df_filtered[
        df_filtered['Change_Type'] == 'Loss'
    ].set_index('Time_Interval')

    # Reindex to ensure proper order on the X-axis
    df_gain = df_gain.reindex(target_intervals)
    df_loss = df_loss.reindex(target_intervals)

    # Rename indices to change the x-axis labels
    rename_mapping = {'Total_Sum': 'Sum', 'Temporal_Extent': 'Extent'}
    df_gain = df_gain.rename(index=rename_mapping)
    df_loss = df_loss.rename(index=rename_mapping)

    # Dynamic Scaling Logic
    max_val_gain = df_gain[metrics_order].sum(axis=1).max()
    max_val_loss = df_loss[metrics_order].abs().sum(axis=1).max()
    max_val = max(max_val_gain, max_val_loss)

    if max_val >= 1_000_000_000_000:
        scale_factor = 1_000_000_000_000
        y_label = "Gross Loss and Gross Gain (trillion pixels)"
    elif max_val >= 1_000_000_000:
        scale_factor = 1_000_000_000
        y_label = "Gross Loss and Gross Gain (billion pixels)"
    elif max_val >= 1_000_000:
        scale_factor = 1_000_000
        y_label = "Gross Loss and Gross Gain (million pixels)"
    elif max_val >= 1_000:
        scale_factor = 1_000
        y_label = "Gross Loss and Gross Gain (thousand pixels)"
    elif max_val >= 100:
        scale_factor = 100
        y_label = "Gross Loss and Gross Gain (hundred pixels)"
    else:
        scale_factor = 1
        y_label = "Gross Loss and Gross Gain (pixels)"

    # Scale the data
    df_gain[metrics_order] = df_gain[metrics_order] / scale_factor
    df_loss[metrics_order] = df_loss[metrics_order] / scale_factor

    max_val_scaled = max_val / scale_factor

    # 2. Color Definition (Consistent with interval charts)
    colors_gain = [
        '#08306b',  # Gain Agreement (Hits) - EXACT MATCH
        '#08519c',  # Gain Space Difference - EXACT MATCH
        '#4292c6',  # Gain Time Difference - NEW
        '#6baed6',  # Gain False Alarms - EXACT MATCH
        '#bdd7e7'   # Gain Misses - EXACT MATCH
    ]

    colors_loss = [
        '#67000d',  # Loss Agreement (Hits) - EXACT MATCH
        '#de2d26',  # Loss Space Difference - EXACT MATCH
        '#ef3b2c',  # Loss Time Difference - NEW
        '#fb6a4a',  # Loss Object-based > Pixel-based (False Alarms) - EXACT MATCH
        '#fcae91'   # Loss Pixel-based > Object-based (Misses) - EXACT MATCH
    ]

    # 3. Plotting the Single Chart
    fig, ax = plt.subplots(figsize=(8, 6))
    fig.subplots_adjust(
        left=0.1,
        right=0.65,
        top=0.90,
        bottom=0.15
    )

    # Plot Gains (Positive)
    df_gain[metrics_order].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=colors_gain,
        edgecolor='none',
        linewidth=0,
        width=0.85,
        legend=False
    )

    # Plot Losses (Negative)
    df_loss[metrics_order].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=colors_loss,
        edgecolor='none',
        linewidth=0,
        width=0.85,
        legend=False
    )

    # Force symmetric Y-axis limits so Y=0 is exactly in the center
    y_limit = max_val_scaled * 1.05
    ax.set_ylim(-y_limit, y_limit)

    # Keep the solid black line at Y=0
    ax.axhline(
        0,
        color='black',
        linewidth=1
    )

    # 4. Custom Legend
    legend_elements = [
        mpatches.Patch(
            color=colors_gain[4],
            label='Gain Pixel-based > Object-based'
        ),
        mpatches.Patch(
            color=colors_gain[3],
            label='Gain Object-based > Pixel-based'
        ),
        mpatches.Patch(
            color=colors_gain[2],
            label='Gain Time Difference'
        ),
        mpatches.Patch(
            color=colors_gain[1],
            label='Gain Space Difference'
        ),
        mpatches.Patch(
            color=colors_gain[0],
            label='Gain Agreement'
        ),
        mpatches.Patch(
            color=colors_loss[0],
            label='Loss Agreement'
        ),
        mpatches.Patch(
            color=colors_loss[1],
            label='Loss Space Difference'
        ),
        mpatches.Patch(
            color=colors_loss[2],
            label='Loss Time Difference'
        ),
        mpatches.Patch(
            color=colors_loss[3],
            label='Loss Object-based > Pixel-based'
        ),
        mpatches.Patch(
            color=colors_loss[4],
            label='Loss Pixel-based > Object-based'
        ),
    ]

    ax.legend(
        handles=legend_elements,
        bbox_to_anchor=(1.02, 0.5),
        loc='center left',
        frameon=False,
        fontsize=12
    )

    # 5. Clean Style and Formatting
    ax.set_ylabel(
        y_label,
        fontsize=14
    )
    ax.set_xlabel('')

    ax.tick_params(
        axis='both',
        which='major',
        labelsize=14
    )
    plt.xticks(
        rotation=0
    )

    # No horizontal or vertical grid lines
    ax.grid(
        False
    )

    # Y-axis formatted with absolute values (no minus sign)
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(
            lambda x, pos: f"{abs(x):g}"
        )
    )

    # Save the figure
    charts_dir = os.path.join(
        output_dir,
        "charts"
    )
    os.makedirs(
        charts_dir,
        exist_ok=True
    )

    output_file = os.path.join(
        charts_dir,
        "gross_change_sum_extent_chart.png"
    )

    plt.savefig(
        output_file,
        dpi=300,
        format='png'
    )
    print(f"Chart successfully saved to: {output_file}\n")

    plt.show()

# Define the CSV path explicitly
tables_dir = os.path.join(
    output_path,
    "tables"
)
change_csv_path = os.path.join(
    tables_dir,
    "change_assessment_metrics.csv"
)

# Execute the plotting function
plot_sum_and_extent_chart(
    csv_path=change_csv_path,
    output_dir=output_path
)


### 4.2 Compute Net Change

In [ ]:
def compute_net_sums_block(
    arr_x_t0: np.ndarray,
    arr_x_t1: np.ndarray,
    arr_y_t0: np.ndarray,
    arr_y_t1: np.ndarray,
    no_data_val: int,
    presence_val: int
) -> Tuple[int, int, int, int]:
    """
    Computes total sums of gains and losses for a single spatial block.

    Parameters
    ----------
    arr_x_t0 : np.ndarray
        Array for Time Series X at time t-1.
    arr_x_t1 : np.ndarray
        Array for Time Series X at time t.
    arr_y_t0 : np.ndarray
        Array for Time Series Y at time t-1.
    arr_y_t1 : np.ndarray
        Array for Time Series Y at time t.
    no_data_val : int
        Pixel value indicating no data.
    presence_val : int
        Pixel value indicating presence.

    Returns
    -------
    Tuple[int, int, int, int]
        Sum of Gain X, Gain Y, Loss X, and Loss Y.
    """
    valid_mask = (
        (arr_x_t0 != no_data_val) & (arr_x_t1 != no_data_val) &
        (arr_y_t0 != no_data_val) & (arr_y_t1 != no_data_val)
    )

    x0 = np.where((arr_x_t0 == presence_val) & valid_mask, 1, 0)
    x1 = np.where((arr_x_t1 == presence_val) & valid_mask, 1, 0)
    y0 = np.where((arr_y_t0 == presence_val) & valid_mask, 1, 0)
    y1 = np.where((arr_y_t1 == presence_val) & valid_mask, 1, 0)

    gain_x = np.maximum(0, x1 - x0)
    gain_y = np.maximum(0, y1 - y0)
    loss_x = np.minimum(0, x1 - x0)
    loss_y = np.minimum(0, y1 - y0)

    return (
        int(np.sum(gain_x)),
        int(np.sum(gain_y)),
        int(np.sum(loss_x)),
        int(np.sum(loss_y))
    )

def calculate_net_change_assessment(
    ts_x_paths: List[str],
    ts_y_paths: List[str],
    no_data_val: int = 255,
    presence_val: int = 1,
    output_csv: str = "net_change_assessment_metrics.csv"
) -> pd.DataFrame:
    """
    Executes Net Change Assessment (Quantity Change) over time series.

    Parameters
    ----------
    ts_x_paths : List[str]
        Paths to Time Series X.
    ts_y_paths : List[str]
        Paths to Time Series Y.
    no_data_val : int, optional
        Value for invalid pixels, by default 255.
    presence_val : int, optional
        Value for presence, by default 1.
    output_csv : str, optional
        Path for output CSV, by default "net_change_metrics.csv".

    Returns
    -------
    pd.DataFrame
        Net Change metrics configured for a diverging stacked chart.
    """
    results = []

    total_qg_x, total_qg_y = 0, 0
    total_ql_x, total_ql_y = 0, 0
    qgh_total, qlh_total = 0, 0
    qgf_total, qlf_total = 0, 0
    qgm_total, qlm_total = 0, 0

    # ADDED TQDM PROGRESS BAR HERE
    for t in tqdm(range(1, len(ts_x_paths)), desc="Net Change Assessment Intervals"):
        path_x0, path_x1 = ts_x_paths[t-1], ts_x_paths[t]
        path_y0, path_y1 = ts_y_paths[t-1], ts_y_paths[t]

        m0 = re.search(r'\d{4}', path_x0)
        m1 = re.search(r'\d{4}', path_x1)
        interval = f"{m0.group(0)}-{m1.group(0)}" if (m0 and m1) else f"T{t}"

        gx, gy, lx, ly = 0, 0, 0, 0

        with rasterio.open(path_x0) as sx0, rasterio.open(path_x1) as sx1, \
             rasterio.open(path_y0) as sy0, rasterio.open(path_y1) as sy1:
            for _, window in sx0.block_windows():
                arr_x0 = sx0.read(1, window=window)
                arr_x1 = sx1.read(1, window=window)
                arr_y0 = sy0.read(1, window=window)
                arr_y1 = sy1.read(1, window=window)

                gxb, gyb, lxb, lyb = compute_net_sums_block(
                    arr_x0, arr_x1, arr_y0, arr_y1, no_data_val, presence_val
                )
                gx += gxb
                gy += gyb
                lx += lxb
                ly += lyb

        qg_x = max(0, gx + lx)
        qg_y = max(0, gy + ly)
        ql_x = min(0, gx + lx)
        ql_y = min(0, gy + ly)

        total_qg_x += qg_x
        total_qg_y += qg_y
        total_ql_x += ql_x
        total_ql_y += ql_y

        qgh = min(qg_x, qg_y)
        qgf = max(0, qg_y - qg_x)
        qgm = max(0, qg_x - qg_y)

        qlh = max(ql_x, ql_y)
        qlf = min(0, ql_y - ql_x)
        qlm = min(0, ql_x - ql_y)

        qgh_total += qgh
        qgf_total += qgf
        qgm_total += qgm
        qlh_total += qlh
        qlf_total += qlf
        qlm_total += qlm

        results.append({
            'Time_Interval': interval,
            'Change_Type': 'Gain',
            'Hits': qgh,
            'False Alarms': qgf,
            'Misses': qgm,
            'Space Difference': 0,
            'Time Difference': 0,
            'Alternation': 0
        })
        results.append({
            'Time_Interval': interval,
            'Change_Type': 'Loss',
            'Hits': abs(qlh),
            'False Alarms': abs(qlf),
            'Misses': abs(qlm),
            'Space Difference': 0,
            'Time Difference': 0,
            'Alternation': 0
        })

    qgv = min(total_qg_x, total_qg_y) - qgh_total
    qlv = max(total_ql_x, total_ql_y) - qlh_total

    total_qg_fa = max(0, total_qg_y - total_qg_x)
    total_qg_misses = max(0, total_qg_x - total_qg_y)
    total_ql_fa = max(0, abs(total_ql_y) - abs(total_ql_x))
    total_ql_misses = max(0, abs(total_ql_x) - abs(total_ql_y))

    results.append({
        'Time_Interval': 'Total_Sum',
        'Change_Type': 'Gain',
        'Hits': qgh_total,
        'False Alarms': total_qg_fa,
        'Misses': total_qg_misses,
        'Space Difference': 0,
        'Time Difference': qgv,
        'Alternation': 0
    })
    results.append({
        'Time_Interval': 'Total_Sum',
        'Change_Type': 'Loss',
        'Hits': abs(qlh_total),
        'False Alarms': total_ql_fa,
        'Misses': total_ql_misses,
        'Space Difference': 0,
        'Time Difference': abs(qlv),
        'Alternation': 0
    })

    ext_gx, ext_gy, ext_lx, ext_ly = 0, 0, 0, 0
    with rasterio.open(ts_x_paths[0]) as sx0, \
         rasterio.open(ts_x_paths[-1]) as sx1, \
         rasterio.open(ts_y_paths[0]) as sy0, \
         rasterio.open(ts_y_paths[-1]) as sy1:
        for _, window in sx0.block_windows():
            ax0 = sx0.read(1, window=window)
            ax1 = sx1.read(1, window=window)
            ay0 = sy0.read(1, window=window)
            ay1 = sy1.read(1, window=window)
            gxb, gyb, lxb, lyb = compute_net_sums_block(
                ax0, ax1, ay0, ay1, no_data_val, presence_val
            )
            ext_gx += gxb
            ext_gy += gyb
            ext_lx += lxb
            ext_ly += lyb

    ext_qg_x = max(0, ext_gx + ext_lx)
    ext_qg_y = max(0, ext_gy + ext_ly)
    ext_ql_x = min(0, ext_gx + ext_lx)
    ext_ql_y = min(0, ext_gy + ext_ly)

    ext_qgh = min(ext_qg_x, ext_qg_y)
    ext_qgf = max(0, ext_qg_y - ext_qg_x)
    ext_qgm = max(0, ext_qg_x - ext_qg_y)

    ext_qlh = max(ext_ql_x, ext_ql_y)
    ext_qlf = min(0, ext_ql_y - ext_ql_x)
    ext_qlm = min(0, ext_ql_x - ext_ql_y)

    results.append({
        'Time_Interval': 'Temporal_Extent',
        'Change_Type': 'Gain',
        'Hits': ext_qgh,
        'False Alarms': ext_qgf,
        'Misses': ext_qgm,
        'Space Difference': 0,
        'Time Difference': 0,
        'Alternation': 0
    })
    results.append({
        'Time_Interval': 'Temporal_Extent',
        'Change_Type': 'Loss',
        'Hits': abs(ext_qlh),
        'False Alarms': abs(ext_qlf),
        'Misses': abs(ext_qlm),
        'Space Difference': 0,
        'Time Difference': 0,
        'Alternation': 0
    })

    df = pd.DataFrame(results)

    metrics = ['Hits', 'False Alarms', 'Misses', 'Space Difference']
    for c_type in ['Gain', 'Loss']:
        mask_tot = (df['Time_Interval'] == 'Total_Sum') & \
                   (df['Change_Type'] == c_type)
        mask_ext = (df['Time_Interval'] == 'Temporal_Extent') & \
                   (df['Change_Type'] == c_type)

        for col in metrics:
            v_tot = df.loc[mask_tot, col].values[0]
            v_ext = df.loc[mask_ext, col].values[0]
            df.loc[mask_tot, 'Alternation'] += abs(v_tot - v_ext)

    df.to_csv(output_csv, index=False)
    return df


net_change_csv_path = os.path.join(
    tables_dir,
    "net_change_assessment_metrics.csv"
)
print(f"Executing Net Change Assessment...\nOutput: {net_change_csv_path}\n")

df_net_change = calculate_net_change_assessment(
    ts_x_paths=time_series_x,
    ts_y_paths=time_series_y,
    no_data_val=no_data_value,
    presence_val=presence_value,
    output_csv=net_change_csv_path
)

display(df_net_change)


#### 4.2.1 Plot Net Change

In [ ]:
def plot_net_change_chart(csv_path: str, gross_csv_path: str, output_dir: str) -> None:
    """
    Reads the change assessment metrics CSV and generates a single diverging
    stacked bar chart for Gains (positive) and Losses (negative) per time interval.

    This function filters out aggregated metrics ('Total_Sum' and 'Temporal_Extent')
    and plots the net change components using a custom blue color palette for
    Gains and a red color palette for Losses. The Y-axis is scaled to match the
    Gross Change chart for direct visual comparison. The final chart is saved as a
    high-resolution PNG file.

    Parameters
    ----------
    csv_path : str
        Path to the CSV file containing the computed NET change assessment metrics.
    gross_csv_path : str
        Path to the CSV file containing the computed GROSS change assessment metrics.
    output_dir : str
        Directory where the 'charts' subfolder will be created and the
        resulting plot will be saved.

    Returns
    -------
    None
    """
    # 1. Prepare Data
    df = pd.read_csv(csv_path)
    df_gross = pd.read_csv(gross_csv_path)

    # Filter out Total_Sum and Temporal_Extent
    df_filtered = df[
        ~df['Time_Interval'].isin(['Total_Sum', 'Temporal_Extent'])
    ].copy()

    df_gross_filtered = df_gross[
        ~df_gross['Time_Interval'].isin(['Total_Sum', 'Temporal_Extent'])
    ].copy()

    # Drop columns not needed for this chart
    if 'Time Difference' in df_filtered.columns:
        df_filtered = df_filtered.drop(columns=['Time Difference'])
    if 'Alternation' in df_filtered.columns:
        df_filtered = df_filtered.drop(columns=['Alternation'])

    # Isolate metrics and invert Losses for the diverging chart
    metrics_order = [
        'Hits',
        'Space Difference',
        'False Alarms',
        'Misses'
    ]

    loss_mask = df_filtered['Change_Type'] == 'Loss'
    df_filtered.loc[loss_mask, metrics_order] *= -1

    # Separate into Gain and Loss for plotting on the same axis
    df_gain = df_filtered[
        df_filtered['Change_Type'] == 'Gain'
    ].set_index('Time_Interval')

    df_loss = df_filtered[
        df_filtered['Change_Type'] == 'Loss'
    ].set_index('Time_Interval')

    # Dynamic Scaling Logic based on GROSS data to sync Y-axis limits
    max_val_gain_gross = df_gross_filtered[df_gross_filtered['Change_Type'] == 'Gain'][metrics_order].sum(axis=1).max()
    max_val_loss_gross = df_gross_filtered[df_gross_filtered['Change_Type'] == 'Loss'][metrics_order].sum(axis=1).max()
    max_val_gross = max(max_val_gain_gross, max_val_loss_gross)

    if max_val_gross >= 1_000_000_000_000:
        scale_factor = 1_000_000_000_000
        y_label = "Net Loss and Net Gain (trillion pixels)"
    elif max_val_gross >= 1_000_000_000:
        scale_factor = 1_000_000_000
        y_label = "Net Loss and Net Gain (billion pixels)"
    elif max_val_gross >= 1_000_000:
        scale_factor = 1_000_000
        y_label = "Net Loss and Net Gain (million pixels)"
    elif max_val_gross >= 1_000:
        scale_factor = 1_000
        y_label = "Net Loss and Net Gain (thousand pixels)"
    elif max_val_gross >= 100:
        scale_factor = 100
        y_label = "Net Loss and Net Gain (hundred pixels)"
    else:
        scale_factor = 1
        y_label = "Net Loss and Net Gain (pixels)"

    # Scale the NET data
    df_gain[metrics_order] = df_gain[metrics_order] / scale_factor
    df_loss[metrics_order] = df_loss[metrics_order] / scale_factor

    # Scaled max value for invisible limits
    max_val_scaled_gross = max_val_gross / scale_factor

    # 2. Styling and Colors
    # 4 shades of Blue for Gains (Dark to Light)
    colors_gain = [
        '#08306b',  # Gain Agreement (Hits)
        '#08519c',  # Gain Space Difference
        '#6baed6',  # Gain False Alarms
        '#bdd7e7'   # Gain Misses
    ]

    # 4 shades of Red for Losses (Dark to Light)
    colors_loss = [
        '#67000d',  # Loss Agreement (Hits)
        '#de2d26',  # Loss Space Difference
        '#fb6a4a',  # Loss Time Series Y > Time Series X (False Alarms)
        '#fcae91'   # Loss Time Series X > Time Series Y (Misses)
    ]

    # 3. Generate Chart (Single Axis)
    # FIXED FIGURE SIZE AND MARGINS
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.subplots_adjust(
        left=0.1,
        right=0.60,
        top=0.90,
        bottom=0.15
    )

    # Plot Gains (Positive)
    df_gain[metrics_order].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=colors_gain,
        edgecolor='none',
        linewidth=0,
        width=0.85,
        legend=False
    )

    # Plot Losses (Negative)
    df_loss[metrics_order].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=colors_loss,
        edgecolor='none',
        linewidth=0,
        width=0.85,
        legend=False
    )

    # Force symmetric Y-axis limits based on GROSS data so Y=0 is exactly in the center
    y_limit = max_val_scaled_gross * 1.05
    ax.set_ylim(-y_limit, y_limit)

    # Add a solid black line at Y=0 to clearly separate gains and losses
    ax.axhline(0, color='black', linewidth=1)

    # 4. Custom Legend
    # Calculate sums to filter out empty categories
    gain_sums = df_gain[metrics_order].sum()
    loss_sums = df_loss[metrics_order].abs().sum()

    legend_elements = []
    if gain_sums['Misses'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[3], label='Gain Pixel-based > Object-based'))
    if gain_sums['False Alarms'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[2], label='Gain Object-based > Pixel-based'))
    if gain_sums['Space Difference'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[1], label='Gain Space Difference'))
    if gain_sums['Hits'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[0], label='Gain Agreement'))

    if loss_sums['Hits'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[0], label='Loss Agreement'))
    if loss_sums['Space Difference'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[1], label='Loss Space Difference'))
    if loss_sums['False Alarms'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[2], label='Loss Object-based > Pixel-based'))
    if loss_sums['Misses'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[3], label='Loss Pixel-based > Object-based'))

    ax.legend(
        handles=legend_elements,
        bbox_to_anchor=(1.02, 0.5),
        loc='center left',
        frameon=False,
        fontsize=12
    )

    # 5. Styling and Formatting Clean
    ax.set_title(
        'Net Gain and Loss During Time Interval',
        fontsize=14,
        pad=10
    )
    ax.set_ylabel(
        y_label,
        fontsize=14
    )

    # Forçar o rótulo do eixo X a ficar em branco (sobrescreve o padrão do pandas)
    ax.set_xlabel('')

    ax.tick_params(
        axis='both',
        which='major',
        labelsize=14
    )
    plt.xticks(rotation=45)

    # Disable grid lines for a clean background
    ax.grid(False)

    # Format Y-axis to show absolute values (remove minus signs)
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(lambda x, pos: f"{abs(x):g}")
    )

    # Remove tight_layout as it conflicts with subplots_adjust
    # plt.tight_layout()

    # Save the figure
    charts_dir = os.path.join(
        output_dir,
        "charts"
    )
    os.makedirs(
        charts_dir,
        exist_ok=True
    )

    output_file = os.path.join(
        charts_dir,
        "net_change_intervals_chart.png"
    )

    # Remove bbox_inches='tight' to preserve exact figure size
    plt.savefig(
        output_file,
        dpi=300,
        format='png'
    )
    print(f"Chart successfully saved to: {output_file}\n")

    plt.show()

# Define the CSV paths explicitly
tables_dir = os.path.join(
    output_path,
    "tables"
)
net_change_csv_path = os.path.join(
    tables_dir,
    "net_change_assessment_metrics.csv"
)
change_csv_path = os.path.join(
    tables_dir,
    "change_assessment_metrics.csv"
)

# Execute the plotting function
plot_net_change_chart(net_change_csv_path, change_csv_path, output_path)


#### 4.2.2 Plot Net Sum and Extent

In [ ]:
def plot_net_sum_and_extent_chart(
    csv_path: str,
    gross_csv_path: str,
    output_dir: str
) -> None:
    """
    Reads the change assessment metrics CSV and generates a single
    diverging stacked bar chart for 'Total_Sum' and 'Temporal_Extent'.

    The Y-axis is scaled to match the Gross Change chart for direct
    visual comparison.

    Parameters
    ----------
    csv_path : str
        Path to the CSV file containing the computed NET change assessment metrics.
    gross_csv_path : str
        Path to the CSV file containing the computed GROSS change assessment metrics.
    output_dir : str
        Directory where the 'charts' subfolder will be created and the
        resulting plot will be saved.

    Returns
    -------
    None
    """
    # 1. Prepare Data
    df = pd.read_csv(csv_path)
    df_gross = pd.read_csv(gross_csv_path)

    # Filter to keep ONLY 'Total_Sum' and 'Temporal_Extent'
    target_intervals = ['Total_Sum', 'Temporal_Extent']
    df_filtered = df[
        df['Time_Interval'].isin(target_intervals)
    ].copy()

    df_gross_filtered = df_gross[
        df_gross['Time_Interval'].isin(target_intervals)
    ].copy()

    # Isolate the 5 specific metrics for stacking
    metrics_order = [
        'Hits',
        'Space Difference',
        'Time Difference',
        'False Alarms',
        'Misses'
    ]

    # Invert Losses to negative values for the diverging chart
    loss_mask = df_filtered['Change_Type'] == 'Loss'
    df_filtered.loc[loss_mask, metrics_order] *= -1

    # Separate into Gain and Loss for plotting on the same axis
    df_gain = df_filtered[
        df_filtered['Change_Type'] == 'Gain'
    ].set_index('Time_Interval')

    df_loss = df_filtered[
        df_filtered['Change_Type'] == 'Loss'
    ].set_index('Time_Interval')

    # Reindex to ensure proper order on the X-axis
    df_gain = df_gain.reindex(target_intervals)
    df_loss = df_loss.reindex(target_intervals)

    # Rename indices to change the x-axis labels
    rename_mapping = {'Total_Sum': 'Sum', 'Temporal_Extent': 'Extent'}
    df_gain = df_gain.rename(index=rename_mapping)
    df_loss = df_loss.rename(index=rename_mapping)

    # Dynamic Scaling Logic based on GROSS data to sync Y-axis limits
    max_val_gain_gross = df_gross_filtered[df_gross_filtered['Change_Type'] == 'Gain'][metrics_order].sum(axis=1).max()
    max_val_loss_gross = df_gross_filtered[df_gross_filtered['Change_Type'] == 'Loss'][metrics_order].sum(axis=1).max()
    max_val_gross = max(max_val_gain_gross, max_val_loss_gross)

    if max_val_gross >= 1_000_000_000_000:
        scale_factor = 1_000_000_000_000
        y_label = "Net Loss and Net Gain (trillion pixels)"
    elif max_val_gross >= 1_000_000_000:
        scale_factor = 1_000_000_000
        y_label = "Net Loss and Net Gain (billion pixels)"
    elif max_val_gross >= 1_000_000:
        scale_factor = 1_000_000
        y_label = "Net Loss and Net Gain (million pixels)"
    elif max_val_gross >= 1_000:
        scale_factor = 1_000
        y_label = "Net Loss and Net Gain (thousand pixels)"
    elif max_val_gross >= 100:
        scale_factor = 100
        y_label = "Net Loss and Net Gain (hundred pixels)"
    else:
        scale_factor = 1
        y_label = "Net Loss and Net Gain (pixels)"

    # Scale the NET data
    df_gain[metrics_order] = df_gain[metrics_order] / scale_factor
    df_loss[metrics_order] = df_loss[metrics_order] / scale_factor

    # Scaled max value for symmetric limits
    max_val_scaled_gross = max_val_gross / scale_factor

    # 2. Definição de Cores (Azul vs Vermelho)
    # 5 shades of Blue for Gains (Dark to Light)
    colors_gain = [
        '#08306b',  # Gain Hits
        '#08519c',  # Gain Space Difference
        '#4292c6',  # Gain Time Difference
        '#6baed6',  # Gain False Alarms
        '#bdd7e7'   # Gain Misses
    ]

    # 5 shades of Red for Losses (Dark to Light)
    colors_loss = [
        '#67000d',  # Loss Hits
        '#a50f15',  # Loss Space Difference
        '#de2d26',  # Loss Time Difference
        '#fb6a4a',  # Loss Time Series Y > Time Series X (False Alarms)
        '#fcae91'   # Loss Time Series X > Time Series Y (Misses)
    ]

    # 3. Plotagem do Gráfico Único
    fig, ax = plt.subplots(figsize=(8, 6))
    fig.subplots_adjust(
        left=0.1,
        right=0.65,
        top=0.90,
        bottom=0.15
    )

    # Plot Gains (Positive)
    df_gain[metrics_order].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=colors_gain,
        edgecolor='none',
        linewidth=0,
        width=0.85,
        legend=False
    )

    # Plot Losses (Negative)
    df_loss[metrics_order].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=colors_loss,
        edgecolor='none',
        linewidth=0,
        width=0.85,
        legend=False
    )

    # Force symmetric Y-axis limits based on GROSS data so Y=0 is exactly in the center
    y_limit = max_val_scaled_gross * 1.05
    ax.set_ylim(-y_limit, y_limit)

    # Mantenha a linha preta sólida em Y=0
    ax.axhline(
        0,
        color='black',
        linewidth=1
    )

    # 4. Legenda Customizada (Matplotlib Patch)
    # Calculate sums to filter out empty categories
    gain_sums = df_gain[metrics_order].sum()
    loss_sums = df_loss[metrics_order].abs().sum()

    legend_elements = []
    if gain_sums['Misses'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[4], label='Gain Pixel-based > Object-based'))
    if gain_sums['False Alarms'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[3], label='Gain Object-based > Pixel-based'))
    if gain_sums['Time Difference'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[2], label='Gain Time Difference'))
    if gain_sums['Space Difference'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[1], label='Gain Space Difference'))
    if gain_sums['Hits'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[0], label='Gain Agreement'))

    if loss_sums['Hits'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[0], label='Loss Agreement'))
    if loss_sums['Space Difference'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[1], label='Loss Space Difference'))
    if loss_sums['Time Difference'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[2], label='Loss Time Difference'))
    if loss_sums['False Alarms'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[3], label='Loss Object-based > Pixel-based'))
    if loss_sums['Misses'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[4], label='Loss Pixel-based > Object-based'))

    ax.legend(
        handles=legend_elements,
        bbox_to_anchor=(1.02, 0.5),
        loc='center left',
        frameon=False,
        fontsize=12
    )

    # 5. Estilo e Formatação Limpa
    ax.set_ylabel(
        y_label,
        fontsize=14
    )
    ax.set_xlabel('')

    ax.tick_params(
        axis='both',
        which='major',
        labelsize=14
    )
    plt.xticks(
        rotation=0
    )

    # Sem linhas de grade horizontais ou verticais
    ax.grid(
        False
    )

    # Eixo Y formatado com valores absolutos (sem sinal de menos)
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(
            lambda x, pos: f"{abs(x):g}"
        )
    )

    # Save the figure
    charts_dir = os.path.join(
        output_dir,
        "charts"
    )
    os.makedirs(
        charts_dir,
        exist_ok=True
    )

    output_file = os.path.join(
        charts_dir,
        "net_change_sum_extent_chart.png"
    )

    plt.savefig(
        output_file,
        dpi=300,
        format='png'
    )
    print(f"Chart successfully saved to: {output_file}\n")

    plt.show()

# Define the CSV paths explicitly
tables_dir = os.path.join(
    output_path,
    "tables"
)
net_change_csv_path = os.path.join(
    tables_dir,
    "net_change_assessment_metrics.csv"
)
change_csv_path = os.path.join(
    tables_dir,
    "change_assessment_metrics.csv"
)

# Execute the plotting function
plot_net_sum_and_extent_chart(
    csv_path=net_change_csv_path,
    gross_csv_path=change_csv_path,
    output_dir=output_path
)

In [ ]:
def plot_net_sum_and_extent_chart(
    csv_path: str,
    output_dir: str
) -> None:
    """
    Reads the change assessment metrics CSV and generates a single
    diverging stacked bar chart for 'Total_Sum' and 'Temporal_Extent'.

    Parameters
    ----------
    csv_path : str
        Path to the CSV file containing the computed change assessment metrics.
    output_dir : str
        Directory where the 'charts' subfolder will be created and the
        resulting plot will be saved.

    Returns
    -------
    None
    """
    # 1. Prepare Data
    df = pd.read_csv(csv_path)

    # Filter to keep ONLY 'Total_Sum' and 'Temporal_Extent'
    target_intervals = ['Total_Sum', 'Temporal_Extent']
    df_filtered = df[
        df['Time_Interval'].isin(target_intervals)
    ].copy()

    # Isolate the 5 specific metrics for stacking
    metrics_order = [
        'Hits',
        'Space Difference',
        'Time Difference',
        'False Alarms',
        'Misses'
    ]

    # Invert Losses to negative values for the diverging chart
    loss_mask = df_filtered['Change_Type'] == 'Loss'
    df_filtered.loc[loss_mask, metrics_order] *= -1

    # Separate into Gain and Loss for plotting on the same axis
    df_gain = df_filtered[
        df_filtered['Change_Type'] == 'Gain'
    ].set_index('Time_Interval')

    df_loss = df_filtered[
        df_filtered['Change_Type'] == 'Loss'
    ].set_index('Time_Interval')

    # Reindex to ensure proper order on the X-axis
    df_gain = df_gain.reindex(target_intervals)
    df_loss = df_loss.reindex(target_intervals)

    # Rename indices to change the x-axis labels
    rename_mapping = {'Total_Sum': 'Sum', 'Temporal_Extent': 'Extent'}
    df_gain = df_gain.rename(index=rename_mapping)
    df_loss = df_loss.rename(index=rename_mapping)

    # Dynamic Scaling Logic
    max_val_gain = df_gain[metrics_order].sum(axis=1).max()
    max_val_loss = df_loss[metrics_order].abs().sum(axis=1).max()
    max_val = max(max_val_gain, max_val_loss)

    if max_val >= 1_000_000_000_000:
        scale_factor = 1_000_000_000_000
        y_label = "Net Loss and Net Gain (trillion pixels)"
    elif max_val >= 1_000_000_000:
        scale_factor = 1_000_000_000
        y_label = "Net Loss and Net Gain (billion pixels)"
    elif max_val >= 1_000_000:
        scale_factor = 1_000_000
        y_label = "Net Loss and Net Gain (million pixels)"
    elif max_val >= 1_000:
        scale_factor = 1_000
        y_label = "Net Loss and Net Gain (thousand pixels)"
    elif max_val >= 100:
        scale_factor = 100
        y_label = "Net Loss and Net Gain (hundred pixels)"
    else:
        scale_factor = 1
        y_label = "Net Loss and Net Gain (pixels)"

    # Scale the data
    df_gain[metrics_order] = df_gain[metrics_order] / scale_factor
    df_loss[metrics_order] = df_loss[metrics_order] / scale_factor

    # 2. Color Definition (Consistent with interval charts)
    colors_gain = [
        '#08306b',  # Gain Agreement (Hits) - EXACT MATCH
        '#08519c',  # Gain Space Difference - EXACT MATCH
        '#4292c6',  # Gain Time Difference - NEW
        '#6baed6',  # Gain False Alarms - EXACT MATCH
        '#bdd7e7'   # Gain Misses - EXACT MATCH
    ]

    colors_loss = [
        '#67000d',  # Loss Agreement (Hits) - EXACT MATCH
        '#de2d26',  # Loss Space Difference - EXACT MATCH
        '#ef3b2c',  # Loss Time Difference - NEW
        '#fb6a4a',  # Loss Object-based > Pixel-based (False Alarms) - EXACT MATCH
        '#fcae91'   # Loss Pixel-based > Object-based (Misses) - EXACT MATCH
    ]

    # 3. Plotagem do Gráfico Único
    fig, ax = plt.subplots(figsize=(8, 6))
    fig.subplots_adjust(
        left=0.1,
        right=0.65,
        top=0.90,
        bottom=0.15
    )

    # Plot Gains (Positive)
    df_gain[metrics_order].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=colors_gain,
        edgecolor='none',
        linewidth=0,
        width=0.85,
        legend=False
    )

    # Plot Losses (Negative)
    df_loss[metrics_order].plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=colors_loss,
        edgecolor='none',
        linewidth=0,
        width=0.85,
        legend=False
    )

    # Mantenha a linha preta sólida em Y=0
    ax.axhline(
        0,
        color='black',
        linewidth=1
    )

    # 4. Legenda Customizada (Matplotlib Patch)
    # Calculate sums to filter out empty categories
    gain_sums = df_gain[metrics_order].sum()
    loss_sums = df_loss[metrics_order].abs().sum()

    legend_elements = []
    if gain_sums['Misses'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[4], label='Gain Object-Based > Pixel-Based'))
    if gain_sums['False Alarms'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[3], label='Gain Pixel-Based > Object-Based'))
    if gain_sums['Time Difference'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[2], label='Gain Time Difference'))
    if gain_sums['Space Difference'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[1], label='Gain Space Difference'))
    if gain_sums['Hits'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_gain[0], label='Gain Agreement'))

    if loss_sums['Hits'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[0], label='Loss Agreement'))
    if loss_sums['Space Difference'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[1], label='Loss Space Difference'))
    if loss_sums['Time Difference'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[2], label='Loss Time Difference'))
    if loss_sums['False Alarms'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[3], label='Loss Pixel-Based > Object-Based'))
    if loss_sums['Misses'] > 0:
        legend_elements.append(mpatches.Patch(color=colors_loss[4], label='Loss Object-Based > Pixel-Based'))

    ax.legend(
        handles=legend_elements,
        bbox_to_anchor=(1.02, 0.5),
        loc='center left',
        frameon=False,
        fontsize=12
    )

    # 5. Estilo e Formatação Limpa
    ax.set_ylabel(
        y_label,
        fontsize=14
    )
    ax.set_xlabel('')

    ax.tick_params(
        axis='both',
        which='major',
        labelsize=14
    )
    plt.xticks(
        rotation=0
    )

    # Sem linhas de grade horizontais ou verticais
    ax.grid(
        False
    )

    # Eixo Y formatado com valores absolutos (sem sinal de menos)
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(
            lambda x, pos: f"{abs(x):g}"
        )
    )

    # Save the figure
    charts_dir = os.path.join(
        output_dir,
        "charts"
    )
    os.makedirs(
        charts_dir,
        exist_ok=True
    )

    output_file = os.path.join(
        charts_dir,
        "net_change_sum_extent_chart.png"
    )

    plt.savefig(
        output_file,
        dpi=300,
        format='png'
    )
    print(f"Chart successfully saved to: {output_file}\n")

    plt.show()

# Execute the plotting function
plot_net_sum_and_extent_chart(
    csv_path=net_change_csv_path,
    output_dir=output_path
)

## **5.Spatial Maps**

### 5.1 Presence Maps

##### 5.1.1 Compute Presence Maps

In [ ]:
def generate_presence_maps(
    ts_x_paths: List[str],
    ts_y_paths: List[str],
    output_dir: str,
    no_data_val: int = no_data_value,
    presence_val: int = presence_value
) -> None:
    """
    Generates the three Presence Maps (A, B, and C) based on the
    methodology by Pontius Jr. et al. (Equations 49, 50, 51).

    Parameters
    ----------
    ts_x_paths : List[str]
        List of file paths for Time Series X.
    ts_y_paths : List[str]
        List of file paths for Time Series Y.
    output_dir : str
        Directory where the output maps will be saved.
    no_data_val : int, optional
        Pixel value indicating no data, by default no_data_value.
    presence_val : int, optional
        Pixel value indicating presence, by default presence_value.

    Returns
    -------
    None
    """
    # Ensure output directory exists (create 'rasters' subfolder)
    maps_dir = os.path.join(
        output_dir,
        "rasters"
    )
    os.makedirs(
        maps_dir,
        exist_ok=True
    )

    # Define output file paths
    path_a = os.path.join(
        maps_dir,
        "presence_map_A_hits.tif"
    )
    path_b = os.path.join(
        maps_dir,
        "presence_map_B_difference.tif"
    )
    path_c = os.path.join(
        maps_dir,
        "presence_map_C_temporal_allocation.tif"
    )

    # Open the first file to retrieve metadata and block windows
    with rasterio.open(
        ts_x_paths[0]
    ) as src_ref:
        meta = src_ref.meta.copy()
        windows = [
            window for _, window in src_ref.block_windows()
        ]

    # Update metadata for minimal typing and compression
    meta.update({
        'dtype': rasterio.int16,
        'nodata': no_data_val,
        'compress': 'deflate'
    })

    print(
        "Generating Presence Maps..."
    )
    print(
        f"Map A: {path_a}"
    )
    print(
        f"Map B: {path_b}"
    )
    print(
        f"Map C: {path_c}"
    )

    # Use ExitStack to manage multiple open files efficiently
    with ExitStack() as stack:
        # Open all source readers once
        srcs_x = [
            stack.enter_context(
                rasterio.open(
                    p
                )
            ) for p in ts_x_paths
        ]
        srcs_y = [
            stack.enter_context(
                rasterio.open(
                    p
                )
            ) for p in ts_y_paths
        ]

        # Open output writers
        dst_a = stack.enter_context(
            rasterio.open(
                path_a,
                'w',
                **meta
            )
        )
        dst_b = stack.enter_context(
            rasterio.open(
                path_b,
                'w',
                **meta
            )
        )
        dst_c = stack.enter_context(
            rasterio.open(
                path_c,
                'w',
                **meta
            )
        )

        # Iterate over spatial blocks (windows)
        for window in tqdm(
            windows,
            desc="Processing Spatial Blocks"
        ):
            height, width = window.height, window.width

            sum_hits = np.zeros(
                (height, width),
                dtype=np.int16
            )
            sum_diff = np.zeros(
                (height, width),
                dtype=np.int16
            )
            sum_abs_diff = np.zeros(
                (height, width),
                dtype=np.int16
            )

            # Initialize a valid mask (True means valid pixel)
            valid_mask = np.ones(
                (height, width),
                dtype=bool
            )

            # Iterate over the pre-opened time points
            for sx, sy in zip(
                srcs_x,
                srcs_y
            ):
                arr_x = sx.read(
                    1,
                    window=window
                )
                arr_y = sy.read(
                    1,
                    window=window
                )

                # Update valid mask (must be valid in all time steps)
                valid_mask &= (
                    arr_x != no_data_val
                )
                valid_mask &= (
                    arr_y != no_data_val
                )

                # Normalize arrays: 1 for presence, 0 for absence
                p_x = np.where(
                    arr_x == presence_val,
                    1,
                    0
                ).astype(
                    np.int16
                )
                p_y = np.where(
                    arr_y == presence_val,
                    1,
                    0
                ).astype(
                    np.int16
                )

                # Eq 49 (Hits): MINIMUM(P_x, P_y)
                sum_hits += np.minimum(
                    p_x,
                    p_y
                )

                # Difference (P_y - P_x)
                diff = p_y - p_x

                # Accumulate diff for Eq 50
                sum_diff += diff

                # Accumulate absolute diff for Eq 51
                sum_abs_diff += np.abs(
                    diff
                )

            # Eq 51 (Temporal Allocation Diff): Sum(|P_y-P_x|) - |Sum(P_y-P_x)|
            c_arr = sum_abs_diff - np.abs(
                sum_diff
            )

            # Mask out no_data pixels
            out_a = np.where(
                valid_mask,
                sum_hits,
                no_data_val
            )
            out_b = np.where(
                valid_mask,
                sum_diff,
                no_data_val
            )
            out_c = np.where(
                valid_mask,
                c_arr,
                no_data_val
            )

            # Write blocks to disk
            dst_a.write(
                out_a.astype(
                    np.int16
                ),
                1,
                window=window
            )
            dst_b.write(
                out_b.astype(
                    np.int16
                ),
                1,
                window=window
            )
            dst_c.write(
                out_c.astype(
                    np.int16
                ),
                1,
                window=window
            )

    print(
        "\nPresence Maps generated successfully!"
    )


# Execute the function
generate_presence_maps(
    ts_x_paths=time_series_x,
    ts_y_paths=time_series_y,
    output_dir=output_path,
    no_data_val=no_data_value,
    presence_val=presence_value
)


#### 5.1.2 Plot Presence Maps

In [ ]:
def compute_display_pixel_size_m(
    raster_path: str,
    downsample_divisor: int,
) -> float:
    """
    Compute horizontal resolution in meters per displayed pixel.

    Parameters
    ----------
    raster_path : str
        Path to a raster file used to derive spatial extent and CRS.
    downsample_divisor : int
        Integer factor used to downsample the raster width for display.

    Returns
    -------
    float
        Pixel size in meters for the downsampled display grid.
    """

    with rasterio.open(raster_path) as src:
        left, bottom, right, top = src.bounds
        lat_mid_src = (top + bottom) / 2.0

        to_ll = Transformer.from_crs(
            src.crs,
            "EPSG:4326",
            always_xy=True,
        )
        lon_l, lat_mid = to_ll.transform(
            left,
            lat_mid_src,
        )
        lon_r, _ = to_ll.transform(
            right,
            lat_mid_src,
        )

        geod = Geod(
            ellps="WGS84",
        )
        _, _, width_m = geod.inv(
            lon_l,
            lat_mid,
            lon_r,
            lat_mid,
        )

        cols_disp = max(
            1,
            src.width // downsample_divisor,
        )

        return width_m / cols_disp


##### 5.1.2.1 Presence Agreement

In [ ]:
def plot_presence_hits_map(
    output_dir: str,
    nodata_val: int,
    raster_filename: str = "presence_map_A_hits.tif",
) -> None:
    """
    Plot the Presence Hits map using a discrete integer scale.

    The visualization uses a specific color logic where the value 0 is
    represented in gray, while values greater than or equal to 1 are
    represented by a gradient color scale (dark blue to yellow) to indicate
    the intensity of the hits.

    Parameters
    ----------
    output_dir : str
        The directory path containing the input raster file and where
        the output image will be saved.
    nodata_val : int
        The NoData value to mask out.
    raster_filename : str, optional
        The filename of the raster to be plotted (default is
        "presence_map_A_hits.tif").

    Returns
    -------
    None
        The function saves a PNG image to the output directory and
        displays the plot.

    Raises
    ------
    FileNotFoundError
        If the specified raster file does not exist in the output directory.
    """
    # 1. Input Validation and Path Setup
    rasters_dir = os.path.join(
        output_dir,
        "rasters",
    )
    raster_path = os.path.join(
        rasters_dir,
        raster_filename,
    )

    if not os.path.exists(
        raster_path,
    ):
        raise FileNotFoundError(
            f"Raster not found: {raster_path}",
        )

    print(
        f"Reading Presence Hits map for plotting: {raster_path}",
    )

    pixel_size_m = compute_display_pixel_size_m(
        raster_path=raster_path,
        downsample_divisor=1
    )

    # 2. Data Loading and Masking
    with rasterio.open(
        raster_path,
    ) as src:
        scale_factor = 1
        data = src.read(
            1,
            out_shape=(
                int(src.height * scale_factor),
                int(src.width * scale_factor),
            ),
            resampling=rasterio.enums.Resampling.nearest,
        )

        # Force masking using the provided variable
        data_masked = np.ma.masked_equal(
            data,
            nodata_val,
        )

        left, bottom, right, top = src.bounds
        src_crs = src.crs
        transform = src.transform

    # Determine the integer range for the color scale
    data_max = int(
        data_masked.max(),
    )

    # Ensure at least 1 to prevent errors if the map is empty/flat
    if data_max == 0:
        data_max = 1

    # 3. Discrete Colormap Configuration
    # 'cividis' goes from dark blue to yellow perfectly
    original_cmap = plt.get_cmap("cividis")

    colors_list = ["#c0c0c0"] + [
        original_cmap(i) for i in np.linspace(0, 1, data_max)
    ]

    cmap = ListedColormap(
        colors_list,
    )

    # 4. Define Boundaries (Bins) for Integers
    # Create boundaries for every integer from 0 to max + 1
    bounds = np.arange(
        0,
        data_max + 2,
    )
    norm = BoundaryNorm(
        bounds,
        cmap.N,
    )

    # 5. Plotting the Figure
    # FIXED FIGURE SIZE AND MARGINS FOR PERFECT OVERLAY
    fig, ax = plt.subplots(
        figsize=(8, 5),
        dpi=300,
    )
    fig.subplots_adjust(
        left=0.1,
        right=0.65,
        top=0.90,
        bottom=0.05
    )

    im = ax.imshow(
        data_masked,
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
    )

    # 6. Legend Configuration
    legend_elements = []

    # Extract unique values actually present in the masked raster data
    present_values = np.unique(
        data_masked.compressed()
    )

    for i in range(0, data_max + 1):
        # Append to legend ONLY if the value is present in the map
        if i in present_values:
            legend_elements.append(
                Patch(
                    facecolor=cmap(norm(i)),
                    edgecolor="none",
                    linewidth=0,
                    label=str(i),
                ),
            )

    ax.legend(
        handles=legend_elements,
        title="Number of Years in Agreement",
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=False,
        fontsize=8,
        title_fontsize=10,
        alignment="left",
        handlelength=2.0,
        handleheight=1.5,
    )

    # 7. Cartographic Elements
    scalebar = ScaleBar(
        pixel_size_m,
        units="m",
        length_fraction=0.35,
        location="lower left",
        box_alpha=0.0
    )
    ax.add_artist(
        scalebar,
    )

    # 8. North Arrow
    north_arrow(
        ax,
        location="upper right",
        shadow=False,
        rotation={
            "degrees": 0,
        },
        scale=0.3
    )

    # 8) Axes styling (with Lat/Lon labels)
    ax.set_title(
        "Presence Agreement",
        fontsize=18,
        pad=5,
    )
    ax.set_aspect(
        "equal",
    )

    # Initialize Transformer (Native CRS -> Lat/Lon)
    to_latlon = Transformer.from_crs(
        src_crs,
        "EPSG:4326",
        always_xy=True,
    )

    # Get dimensions from the loaded data
    height, width = data.shape

    def format_lon(
        x,
        pos,
    ):
        """
        Convert a pixel column index to a formatted Longitude string.
        """
        # Clamp x to be within image bounds
        x = np.clip(
            x,
            0,
            width - 1,
        )
        # Project pixel to map coordinates (using middle of height)
        x_proj, y_proj = rasterio.transform.xy(
            transform,
            height // 2,
            x,
        )
        lon, lat = to_latlon.transform(
            x_proj,
            y_proj,
        )
        return f"{lon:.1f}°"

    def format_lat(
        y,
        pos,
    ):
        """
        Convert a pixel row index to a formatted Latitude string.
        """
        # Clamp y to be within image bounds
        y = np.clip(
            y,
            0,
            height - 1,
        )
        # Project pixel to map coordinates (using middle of width)
        x_proj, y_proj = rasterio.transform.xy(
            transform,
            y,
            width // 2,
        )
        lon, lat = to_latlon.transform(
            x_proj,
            y_proj,
        )
        return f"{lat:.1f}°"

    # Apply the custom formatters
    ax.xaxis.set_major_formatter(
        FuncFormatter(
            format_lon,
        ),
    )
    ax.yaxis.set_major_formatter(
        FuncFormatter(
            format_lat,
        ),
    )

    # Limit ticks to avoid overcrowding
    ax.xaxis.set_major_locator(
        mticker.MaxNLocator(
            nbins=4,
        ),
    )
    ax.yaxis.set_major_locator(
        mticker.MaxNLocator(
            nbins=6,
        ),
    )

    # Final styling for ticks
    ax.tick_params(
        axis="both",
        which="major",
        labelsize=7,
        pad=4,
    )
    plt.setp(
        ax.get_yticklabels(),
        rotation=90,
        va="center",
    )

    maps_dir = os.path.join(
        output_dir,
        "maps",
    )
    os.makedirs(
        maps_dir,
        exist_ok=True,
    )

    output_figure_path = os.path.join(
        maps_dir,
        "map_presence_hits.png",
    )

    # NO BBOX_INCHES TO PRESERVE EXACT FIGURE SIZE
    plt.savefig(
        output_figure_path,
        dpi=300,
        format="png",
    )
    plt.show()

    print(
        f"Map successfully saved to: {output_figure_path}",
    )

# Execute the function
plot_presence_hits_map(
    output_dir=output_path,
    nodata_val=no_data_value,
    raster_filename="presence_map_A_hits.tif",
)


##### 5.1.2.2 Presence Difference

In [ ]:
def plot_presence_difference_map(
    output_dir: str,
    nodata_val: int,
    raster_filename: str = "presence_map_B_difference.tif",
) -> None:
    """
    Plot the Presence Difference map using a diverging discrete integer scale.

    0 is represented in gray.
    Positive values use a Blue gradient (darkest at max, lightest near 0).
    Negative values use a Red gradient (darkest at min, lightest near 0).

    Parameters
    ----------
    output_dir : str
        Directory where the 'rasters' and 'maps' folders are located.
    nodata_val : int
        Pixel value indicating no data.
    raster_filename : str, optional
        Filename of the raster to be plotted. Default is "presence_map_B_difference.tif".

    Returns
    -------
    None
    """
    # 1. Input Validation and Path Setup
    rasters_dir = os.path.join(
        output_dir,
        "rasters"
    )
    raster_path = os.path.join(
        rasters_dir,
        raster_filename
    )

    if not os.path.exists(
        raster_path
    ):
        raise FileNotFoundError(
            f"Raster not found: {raster_path}"
        )

    print(
        f"Reading Presence Difference map for plotting: {raster_path}"
    )

    pixel_size_m = compute_display_pixel_size_m(
        raster_path=raster_path,
        downsample_divisor=1
    )

    # 2. Data Loading and Masking
    with rasterio.open(
        raster_path
    ) as src:
        scale_factor = 1
        data = src.read(
            1,
            out_shape=(
                int(
                    src.height * scale_factor
                ),
                int(
                    src.width * scale_factor
                ),
            ),
            resampling=rasterio.enums.Resampling.nearest,
        )

        data_masked = np.ma.masked_equal(
            data,
            nodata_val
        )

        left, bottom, right, top = src.bounds
        src_crs = src.crs
        transform = src.transform

    # 3. Color Map and Range Configuration
    data_max = int(
        data_masked.max()
    )
    data_min = int(
        data_masked.min()
    )

    if data_max < 1:
        data_max = 1
    if data_min > -1:
        data_min = -1

    colors_dict = {
        0: "#c0c0c0"
    }  # Gray for 0

    red_cmap = plt.get_cmap(
        "Reds"
    )
    red_colors = red_cmap(
        np.linspace(
            0.2,
            0.9,
            abs(
                data_min
            )
        )
    )
    for i, val in enumerate(
        range(
            -1,
            data_min - 1,
            -1
        )
    ):
        colors_dict[val] = red_colors[i]

    blue_cmap = plt.get_cmap(
        "Blues"
    )
    blue_colors = blue_cmap(
        np.linspace(
            0.2,
            0.9,
            data_max
        )
    )
    for i, val in enumerate(
        range(
            1,
            data_max + 1
        )
    ):
        colors_dict[val] = blue_colors[i]

    all_vals = np.arange(
        data_min,
        data_max + 1
    )
    colors_list = [
        colors_dict.get(
            v,
            "#000000"
        ) for v in all_vals
    ]

    cmap = ListedColormap(
        colors_list
    )

    bounds = np.arange(
        data_min - 0.5,
        data_max + 1.5,
        1
    )
    norm = BoundaryNorm(
        bounds,
        cmap.N
    )

    # 4. Plotting the Figure
    fig, ax = plt.subplots(
        figsize=(8, 5),
        dpi=300,
    )
    fig.subplots_adjust(
        left=0.1,
        right=0.65,
        top=0.90,
        bottom=0.05
    )

    im = ax.imshow(
        data_masked,
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
    )

    # 5. Legend Configuration
    legend_elements = []
    present_values = np.unique(
        data_masked.compressed()
    )
    present_values_sorted = np.sort(
        present_values
    )[::-1]

    for val in present_values_sorted:
        legend_elements.append(
            Patch(
                facecolor=colors_dict.get(
                    val,
                    "#000"
                ),
                edgecolor="none",
                linewidth=0,
                label=str(
                    val
                ),
            ),
        )

    ax.legend(
        handles=legend_elements,
        title="Accumulated Difference",
        loc="center left",
        bbox_to_anchor=(
            1.02,
            0.5
        ),
        frameon=False,
        fontsize=8,
        title_fontsize=10,
        alignment="left",
        handlelength=2.0,
        handleheight=1.5,
    )

    # 6. Cartographic Elements
    scalebar = ScaleBar(
        pixel_size_m,
        units="m",
        length_fraction=0.35,
        location="lower left",
        box_alpha=0.0
    )
    ax.add_artist(
        scalebar
    )

    north_arrow(
        ax,
        location="upper right",
        shadow=False,
        rotation={
            "degrees": 0
        },
        scale=0.3
    )

    # 7. Axes Styling and Final Output
    ax.set_title(
        "Presence Difference",
        fontsize=18,
        pad=5
    )
    ax.set_aspect(
        "equal"
    )

    to_latlon = Transformer.from_crs(
        src_crs,
        "EPSG:4326",
        always_xy=True
    )
    height, width = data.shape

    def format_lon(
        x,
        pos
    ):
        x = np.clip(
            x,
            0,
            width - 1
        )
        x_proj, y_proj = rasterio.transform.xy(
            transform,
            height // 2,
            x
        )
        lon, lat = to_latlon.transform(
            x_proj,
            y_proj
        )
        return f"{lon:.1f}°"

    def format_lat(
        y,
        pos
    ):
        y = np.clip(
            y,
            0,
            height - 1
        )
        x_proj, y_proj = rasterio.transform.xy(
            transform,
            y,
            width // 2
        )
        lon, lat = to_latlon.transform(
            x_proj,
            y_proj
        )
        return f"{lat:.1f}°"

    ax.xaxis.set_major_formatter(
        FuncFormatter(
            format_lon
        )
    )
    ax.yaxis.set_major_formatter(
        FuncFormatter(
            format_lat
        )
    )

    ax.xaxis.set_major_locator(
        mticker.MaxNLocator(
            nbins=4
        )
    )
    ax.yaxis.set_major_locator(
        mticker.MaxNLocator(
            nbins=6
        )
    )

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=7,
        pad=4
    )
    plt.setp(
        ax.get_yticklabels(),
        rotation=90,
        va="center"
    )

    maps_dir = os.path.join(
        output_dir,
        "maps"
    )
    os.makedirs(
        maps_dir,
        exist_ok=True
    )

    output_figure_path = os.path.join(
        maps_dir,
        "map_presence_difference.png"
    )

    plt.savefig(
        output_figure_path,
        dpi=300,
        format="png"
    )
    plt.show()

    print(
        f"Map successfully saved to: {output_figure_path}"
    )

# Execute the function
plot_presence_difference_map(
    output_dir=output_path,
    nodata_val=no_data_value,
    raster_filename="presence_map_B_difference.tif",
)


##### 5.1.2.3 Temporal Allocation of Presence

In [ ]:
def plot_presence_temporal_allocation_map(
    output_dir: str,
    nodata_val: int,
    raster_filename: str = "presence_map_C_temporal_allocation.tif",
) -> None:
    """
    Plot the Presence Temporal Allocation map using a discrete integer scale.

    0 is represented in gray.
    Positive values use a gradient transitioning from dark red (1)
    to yellow, and finally to light green (maximum value).

    Parameters
    ----------
    output_dir : str
        Directory where the 'rasters' and 'maps' folders are located.
    nodata_val : int
        Pixel value indicating no data.
    raster_filename : str, optional
        Filename of the raster to be plotted. Default is "presence_map_C_temporal_allocation.tif".

    Returns
    -------
    None
    """
    # 1. Input Validation and Path Setup
    rasters_dir = os.path.join(
        output_dir,
        "rasters",
    )
    raster_path = os.path.join(
        rasters_dir,
        raster_filename,
    )

    if not os.path.exists(
        raster_path,
    ):
        raise FileNotFoundError(
            f"Raster not found: {raster_path}",
        )

    print(
        f"Reading Presence Temporal Allocation map for plotting: {raster_path}",
    )

    pixel_size_m = compute_display_pixel_size_m(
        raster_path=raster_path,
        downsample_divisor=1
    )

    # 2. Data Loading and Masking
    with rasterio.open(
        raster_path,
    ) as src:
        scale_factor = 1
        data = src.read(
            1,
            out_shape=(
                int(
                    src.height * scale_factor
                ),
                int(
                    src.width * scale_factor
                ),
            ),
            resampling=rasterio.enums.Resampling.nearest,
        )

        # Force masking using the provided variable
        data_masked = np.ma.masked_equal(
            data,
            nodata_val,
        )

        left, bottom, right, top = src.bounds
        src_crs = src.crs
        transform = src.transform

    # 3. Determine range and colors
    data_max = int(
        data_masked.max()
    )

    # Ensure safe bounds just in case the map is entirely 0
    if data_max < 1:
        data_max = 1

    colors_dict = {
        0: "#c0c0c0"
    }  # Gray for 0

    # Positive values: Custom gradient Dark Red -> Yellow -> Light Green
    pos_cmap = LinearSegmentedColormap.from_list(
        "red_yellow_green",
        [
            "#8b0000",
            "#ffff00",
            "#90ee90"
        ]
    )

    for i, val in enumerate(
        range(
            1,
            data_max + 1
        )
    ):
        # Normalize fraction between 0 and 1
        if data_max > 1:
            frac = i / (
                data_max - 1
            )
        else:
            frac = 0.5

        colors_dict[val] = pos_cmap(
            frac
        )

    # Build standard lists for colormap
    all_vals = np.arange(
        0,
        data_max + 1
    )
    colors_list = [
        colors_dict.get(
            v,
            "#000000"
        ) for v in all_vals
    ]

    cmap = ListedColormap(
        colors_list
    )

    # Create boundaries for every integer from 0 to max
    bounds = np.arange(
        -0.5,
        data_max + 1.5,
        1
    )
    norm = BoundaryNorm(
        bounds,
        cmap.N
    )

    # 4. Plotting the Figure
    # FIXED FIGURE SIZE AND MARGINS FOR PERFECT OVERLAY
    fig, ax = plt.subplots(
        figsize=(8, 5),
        dpi=300,
    )
    fig.subplots_adjust(
        left=0.1,
        right=0.65,
        top=0.90,
        bottom=0.05
    )

    im = ax.imshow(
        data_masked,
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
    )

    # 5. Legend Configuration
    legend_elements = []
    present_values = np.unique(
        data_masked.compressed()
    )
    # Sort ascending: 0 -> max positive
    present_values_sorted = np.sort(
        present_values
    )

    for val in present_values_sorted:
        legend_elements.append(
            Patch(
                facecolor=colors_dict.get(
                    val,
                    "#000"
                ),
                edgecolor="none",
                linewidth=0,
                label=str(
                    val
                ),
            ),
        )

    ax.legend(
        handles=legend_elements,
        title="Temporal Allocation",
        loc="center left",
        bbox_to_anchor=(
            1.02,
            0.5
        ),
        frameon=False,
        fontsize=8,
        title_fontsize=10,
        alignment="left",
        handlelength=2.0,
        handleheight=1.5,
    )

    # 6. Cartographic Elements
    scalebar = ScaleBar(
        pixel_size_m,
        units="m",
        length_fraction=0.35,
        location="lower left",
        box_alpha=0.0
    )
    ax.add_artist(
        scalebar
    )

    north_arrow(
        ax,
        location="upper right",
        shadow=False,
        rotation={
            "degrees": 0
        },
        scale=0.3
    )

    # 7. Axes styling (with Lat/Lon labels)
    ax.set_title(
        "Presence Temporal Allocation",
        fontsize=18,
        pad=5,
    )
    ax.set_aspect(
        "equal"
    )

    to_latlon = Transformer.from_crs(
        src_crs,
        "EPSG:4326",
        always_xy=True,
    )
    height, width = data.shape

    def format_lon(
        x,
        pos
    ):
        x = np.clip(
            x,
            0,
            width - 1
        )
        x_proj, y_proj = rasterio.transform.xy(
            transform,
            height // 2,
            x
        )
        lon, lat = to_latlon.transform(
            x_proj,
            y_proj
        )
        return f"{lon:.1f}°"

    def format_lat(
        y,
        pos
    ):
        y = np.clip(
            y,
            0,
            height - 1
        )
        x_proj, y_proj = rasterio.transform.xy(
            transform,
            y,
            width // 2
        )
        lon, lat = to_latlon.transform(
            x_proj,
            y_proj
        )
        return f"{lat:.1f}°"

    ax.xaxis.set_major_formatter(
        FuncFormatter(
            format_lon
        )
    )
    ax.yaxis.set_major_formatter(
        FuncFormatter(
            format_lat
        )
    )

    ax.xaxis.set_major_locator(
        mticker.MaxNLocator(
            nbins=4
        )
    )
    ax.yaxis.set_major_locator(
        mticker.MaxNLocator(
            nbins=6
        )
    )

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=7,
        pad=4
    )
    plt.setp(
        ax.get_yticklabels(),
        rotation=90,
        va="center"
    )

    maps_dir = os.path.join(
        output_dir,
        "maps"
    )
    os.makedirs(
        maps_dir,
        exist_ok=True
    )

    output_figure_path = os.path.join(
        maps_dir,
        "map_presence_temporal_allocation.png",
    )

    # NO BBOX_INCHES TO PRESERVE EXACT FIGURE SIZE
    plt.savefig(
        output_figure_path,
        dpi=300,
        format="png",
    )
    plt.show()

    print(
        f"Map successfully saved to: {output_figure_path}"
    )

# Execute the function
plot_presence_temporal_allocation_map(
    output_dir=output_path,
    nodata_val=no_data_value,
    raster_filename="presence_map_C_temporal_allocation.tif",
)


### 5.2 Change Maps

#### 5.2.1 Compute Change Maps


In [ ]:
def generate_change_maps(
    ts_x_paths: List[str],
    ts_y_paths: List[str],
    output_dir: str,
    no_data_val: int = 255,
    presence_val: int = 1
) -> None:
    """
    Generates the three Change Maps (D, E, and F) based on the
    methodology by Pontius Jr. et al. (Equations 52, 53, 54).

    Parameters
    ----------
    ts_x_paths : List[str]
        List of file paths for Time Series X.
    ts_y_paths : List[str]
        List of file paths for Time Series Y.
    output_dir : str
        Directory where the output maps will be saved.
    no_data_val : int, optional
        Pixel value indicating no data, by default 255.
    presence_val : int, optional
        Pixel value indicating presence, by default 1.

    Returns
    -------
    None
    """
    maps_dir = os.path.join(output_dir, "rasters")
    os.makedirs(maps_dir, exist_ok=True)

    path_d = os.path.join(maps_dir, "change_map_D_hits.tif")
    path_e = os.path.join(maps_dir, "change_map_E_difference.tif")
    path_f = os.path.join(
        maps_dir,
        "change_map_F_temporal_allocation.tif"
    )

    num_times = len(ts_x_paths)

    with rasterio.open(ts_x_paths[0]) as src_ref:
        meta = src_ref.meta.copy()
        windows = [window for _, window in src_ref.block_windows()]

    # Use int16 to accommodate negative values safely and save space
    meta.update({
        'dtype': rasterio.int16,
        'nodata': no_data_val,
        'compress': 'deflate'
    })

    print("Generating Change Maps...")
    print(f"Map D: {path_d}")
    print(f"Map E: {path_e}")
    print(f"Map F: {path_f}")

    # Use ExitStack to manage multiple open files safely and efficiently
    with ExitStack() as stack:
        srcs_x = [
            stack.enter_context(rasterio.open(p)) for p in ts_x_paths
        ]
        srcs_y = [
            stack.enter_context(rasterio.open(p)) for p in ts_y_paths
        ]

        dst_d = stack.enter_context(rasterio.open(path_d, 'w', **meta))
        dst_e = stack.enter_context(rasterio.open(path_e, 'w', **meta))
        dst_f = stack.enter_context(rasterio.open(path_f, 'w', **meta))

        for window in tqdm(windows, desc="Processing Spatial Blocks"):
            height, width = window.height, window.width

            # Accumulators for D_n and F_n components
            d_n_sum = np.zeros((height, width), dtype=np.int16)
            abs_diff_sum = np.zeros((height, width), dtype=np.int16)
            valid_mask = np.ones((height, width), dtype=bool)

            # Read initial time (t=0)
            arr_x0 = srcs_x[0].read(1, window=window)
            arr_y0 = srcs_y[0].read(1, window=window)

            valid_mask &= (arr_x0 != no_data_val) & (arr_y0 != no_data_val)

            p_x0 = np.where(
                arr_x0 == presence_val, 1, 0
            ).astype(np.int16)
            p_y0 = np.where(
                arr_y0 == presence_val, 1, 0
            ).astype(np.int16)

            p_x_prev, p_y_prev = p_x0, p_y0

            # Process intervals
            for t in range(1, num_times):
                arr_x_t = srcs_x[t].read(1, window=window)
                arr_y_t = srcs_y[t].read(1, window=window)

                valid_mask &= (
                    (arr_x_t != no_data_val) & (arr_y_t != no_data_val)
                )

                p_x_curr = np.where(
                    arr_x_t == presence_val, 1, 0
                ).astype(np.int16)
                p_y_curr = np.where(
                    arr_y_t == presence_val, 1, 0
                ).astype(np.int16)

                delta_x = p_x_curr - p_x_prev
                delta_y = p_y_curr - p_y_prev

                gain_x = np.maximum(0, delta_x)
                gain_y = np.maximum(0, delta_y)
                loss_x = np.minimum(0, delta_x)
                loss_y = np.minimum(0, delta_y)

                g_hn = np.minimum(gain_x, gain_y)
                l_hn = np.maximum(loss_x, loss_y)

                # Eq 52: accumulate (Gh_n - Lh_n)
                d_n_sum += (g_hn - l_hn)

                # Eq 54 component: accumulate |Delta_Y - Delta_X|
                abs_diff_sum += np.abs(delta_y - delta_x)

                p_x_prev, p_y_prev = p_x_curr, p_y_curr

            # p_x_prev and p_y_prev are now P_x_final and P_y_final
            # Eq 53: E_n = (Y_final - Y_init) - (X_final - X_init)
            e_n = (p_y_prev - p_y0) - (p_x_prev - p_x0)

            # Eq 54: F_n = Sum(|Delta_Y - Delta_X|) - |E_n|
            f_n = abs_diff_sum - np.abs(e_n)

            # Mask no_data values
            out_d = np.where(valid_mask, d_n_sum, no_data_val)
            out_e = np.where(valid_mask, e_n, no_data_val)
            out_f = np.where(valid_mask, f_n, no_data_val)

            # Write to disk
            dst_d.write(out_d.astype(np.int16), 1, window=window)
            dst_e.write(out_e.astype(np.int16), 1, window=window)
            dst_f.write(out_f.astype(np.int16), 1, window=window)

    print("\nChange Maps generated successfully!")


# Execute the function
generate_change_maps(
    ts_x_paths=time_series_x,
    ts_y_paths=time_series_y,
    output_dir=output_path,
    no_data_val=no_data_value,
    presence_val=presence_value
)


#### 5.2.2 Plot Change Maps

##### 5.2.2.1 Change Agreement

In [ ]:
def plot_change_hits_map(
    output_dir: str,
    nodata_val: int,
    raster_filename: str = "change_map_D_hits.tif",
) -> None:
    """
    Plot the Change Hits map using the turbo_r colormap for positive values.

    0 is represented in gray.
    Positive values use an inverted turbo scale: Dark Red -> Yellow -> Light Blue -> Dark Blue.

    Parameters
    ----------
    output_dir : str
        Directory where the 'rasters' and 'maps' folders are located.
    nodata_val : int
        Pixel value indicating no data.
    raster_filename : str, optional
        Filename of the raster to be plotted. Default is "change_map_D_hits.tif".

    Returns
    -------
    None
    """
    # 1. Input Validation and Path Setup
    rasters_dir = os.path.join(
        output_dir,
        "rasters",
    )
    raster_path = os.path.join(
        rasters_dir,
        raster_filename,
    )

    if not os.path.exists(
        raster_path,
    ):
        raise FileNotFoundError(
            f"Raster not found: {raster_path}",
        )

    print(
        f"Reading Change Hits map for plotting: {raster_path}",
    )

    pixel_size_m = compute_display_pixel_size_m(
        raster_path=raster_path,
        downsample_divisor=1
    )

    # 2. Data Loading and Masking
    with rasterio.open(
        raster_path,
    ) as src:
        scale_factor = 1
        data = src.read(
            1,
            out_shape=(
                int(
                    src.height * scale_factor
                ),
                int(
                    src.width * scale_factor
                ),
            ),
            resampling=rasterio.enums.Resampling.nearest,
        )

        # Force masking using the provided variable
        data_masked = np.ma.masked_equal(
            data,
            nodata_val,
        )

        left, bottom, right, top = src.bounds
        src_crs = src.crs
        transform = src.transform

    # 3. Determine range and colors
    data_max = int(
        data_masked.max()
    )
    data_min = int(
        data_masked.min()
    )

    # Ensure safe bounds
    if data_max < 1:
        data_max = 1
    if data_min > 0:
        data_min = 0

    colors_dict = {
        0: "#c0c0c0"
    }  # Gray for 0

    # Positive values: turbo_r (Dark Red -> Yellow -> Light Blue -> Dark Blue)
    cmap_turbo = plt.get_cmap(
        "turbo_r"
    )
    pos_colors = cmap_turbo(
        np.linspace(
            0.05,
            0.95,
            data_max
        )
    )
    for i, val in enumerate(
        range(
            1,
            data_max + 1
        )
    ):
        colors_dict[val] = pos_colors[i]

    # Just in case there are negative values, handle them smoothly
    if data_min < 0:
        for val in range(
            data_min,
            0
        ):
            colors_dict[val] = "#000000"

    # Build standard lists for colormap
    all_vals = np.arange(
        data_min,
        data_max + 1
    )
    colors_list = [
        colors_dict.get(
            v,
            "#000000"
        ) for v in all_vals
    ]

    cmap = ListedColormap(
        colors_list
    )

    # Create boundaries for every integer from min to max
    bounds = np.arange(
        data_min - 0.5,
        data_max + 1.5,
        1
    )
    norm = BoundaryNorm(
        bounds,
        cmap.N
    )

    # 4. Plotting the Figure
    fig, ax = plt.subplots(
        figsize=(8, 5),
        dpi=300,
    )
    fig.subplots_adjust(
        left=0.1,
        right=0.65,
        top=0.90,
        bottom=0.05
    )

    im = ax.imshow(
        data_masked,
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
    )

    # 5. Legend Configuration
    legend_elements = []
    present_values = np.unique(
        data_masked.compressed()
    )
    # Sort ascending: 0 -> max positive
    present_values_sorted = np.sort(
        present_values
    )

    for val in present_values_sorted:
        legend_elements.append(
            Patch(
                facecolor=colors_dict.get(
                    val,
                    "#000"
                ),
                edgecolor="none",
                linewidth=0,
                label=str(
                    val
                ),
            ),
        )

    ax.legend(
        handles=legend_elements,
        title="Number of Change Agreements",
        loc="center left",
        bbox_to_anchor=(
            1.02,
            0.5
        ),
        frameon=False,
        fontsize=8,
        title_fontsize=10,
        alignment="left",
        handlelength=2.0,
        handleheight=1.5,
    )

    # 6. Cartographic Elements
    scalebar = ScaleBar(
        pixel_size_m,
        units="m",
        length_fraction=0.35,
        location="lower left",
        box_alpha=0.0
    )
    ax.add_artist(
        scalebar
    )

    north_arrow(
        ax,
        location="upper right",
        shadow=False,
        rotation={
            "degrees": 0
        },
        scale=0.3
    )

    # 7. Axes styling (with Lat/Lon labels)
    ax.set_title(
        "Change Agreement",
        fontsize=18,
        pad=5,
    )
    ax.set_aspect(
        "equal"
    )

    to_latlon = Transformer.from_crs(
        src_crs,
        "EPSG:4326",
        always_xy=True,
    )
    height, width = data.shape

    def format_lon(
        x,
        pos
    ):
        x = np.clip(
            x,
            0,
            width - 1
        )
        x_proj, y_proj = rasterio.transform.xy(
            transform,
            height // 2,
            x
        )
        lon, lat = to_latlon.transform(
            x_proj,
            y_proj
        )
        return f"{lon:.1f}°"

    def format_lat(
        y,
        pos
    ):
        y = np.clip(
            y,
            0,
            height - 1
        )
        x_proj, y_proj = rasterio.transform.xy(
            transform,
            y,
            width // 2
        )
        lon, lat = to_latlon.transform(
            x_proj,
            y_proj
        )
        return f"{lat:.1f}°"

    ax.xaxis.set_major_formatter(
        FuncFormatter(
            format_lon
        )
    )
    ax.yaxis.set_major_formatter(
        FuncFormatter(
            format_lat
        )
    )

    ax.xaxis.set_major_locator(
        mticker.MaxNLocator(
            nbins=4
        )
    )
    ax.yaxis.set_major_locator(
        mticker.MaxNLocator(
            nbins=6
        )
    )

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=7,
        pad=4
    )
    plt.setp(
        ax.get_yticklabels(),
        rotation=90,
        va="center"
    )

    maps_dir = os.path.join(
        output_dir,
        "maps"
    )
    os.makedirs(
        maps_dir,
        exist_ok=True
    )

    output_figure_path = os.path.join(
        maps_dir,
        "map_change_hits.png",
    )

    plt.savefig(
        output_figure_path,
        dpi=300,
        format="png",
    )
    plt.show()

    print(
        f"Map successfully saved to: {output_figure_path}"
    )

# Execute the function
plot_change_hits_map(
    output_dir=output_path,
    nodata_val=no_data_value,
    raster_filename="change_map_D_hits.tif",
)


##### 5.2.2.2 Change Difference

In [ ]:
def plot_change_difference_map(
    output_dir: str,
    nodata_val: int,
    raster_filename: str = "change_map_E_difference.tif",
) -> None:
    """
    Plot the Change Difference map using a diverging discrete integer scale.

    0 is represented in gray.
    Positive values use a Blue gradient (darkest at max, lightest near 0).
    Negative values use a Red gradient (darkest at min, lightest near 0).

    Parameters
    ----------
    output_dir : str
        Directory where the 'rasters' and 'maps' folders are located.
    nodata_val : int
        Pixel value indicating no data.
    raster_filename : str, optional
        Filename of the raster to be plotted. Default is "change_map_E_difference.tif".

    Returns
    -------
    None
    """
    # 1. Input Validation and Path Setup
    rasters_dir = os.path.join(
        output_dir,
        "rasters",
    )
    raster_path = os.path.join(
        rasters_dir,
        raster_filename,
    )

    if not os.path.exists(
        raster_path,
    ):
        raise FileNotFoundError(
            f"Raster not found: {raster_path}",
        )

    print(
        f"Reading Change Difference map for plotting: {raster_path}",
    )

    pixel_size_m = compute_display_pixel_size_m(
        raster_path=raster_path,
        downsample_divisor=1
    )

    # 2. Data Loading and Masking
    with rasterio.open(
        raster_path,
    ) as src:
        scale_factor = 1
        data = src.read(
            1,
            out_shape=(
                int(
                    src.height * scale_factor
                ),
                int(
                    src.width * scale_factor
                ),
            ),
            resampling=rasterio.enums.Resampling.nearest,
        )

        # Force masking using the provided variable
        data_masked = np.ma.masked_equal(
            data,
            nodata_val,
        )

        left, bottom, right, top = src.bounds
        src_crs = src.crs
        transform = src.transform

    # 3. Determine range and colors
    data_max = int(
        data_masked.max()
    )
    data_min = int(
        data_masked.min()
    )

    # Ensure safe bounds just in case the map is entirely 0
    if data_max < 1:
        data_max = 1
    if data_min > -1:
        data_min = -1

    colors_dict = {
        0: "#c0c0c0"
    }  # Gray for 0

    # Negative values: Reds (index 0 is light, index -1 is dark)
    red_cmap = plt.get_cmap(
        "Reds"
    )
    red_colors = red_cmap(
        np.linspace(
            0.2,
            0.9,
            abs(
                data_min
            )
        )
    )
    for i, val in enumerate(
        range(
            -1,
            data_min - 1,
            -1
        )
    ):
        colors_dict[val] = red_colors[i]

    # Positive values: Blues (index 0 is light, index -1 is dark)
    blue_cmap = plt.get_cmap(
        "Blues"
    )
    blue_colors = blue_cmap(
        np.linspace(
            0.2,
            0.9,
            data_max
        )
    )
    for i, val in enumerate(
        range(
            1,
            data_max + 1
        )
    ):
        colors_dict[val] = blue_colors[i]

    # Build standard lists for colormap
    all_vals = np.arange(
        data_min,
        data_max + 1
    )
    colors_list = [
        colors_dict.get(
            v,
            "#000000"
        ) for v in all_vals
    ]

    cmap = ListedColormap(
        colors_list
    )

    # Create boundaries for every integer from min to max
    bounds = np.arange(
        data_min - 0.5,
        data_max + 1.5,
        1
    )
    norm = BoundaryNorm(
        bounds,
        cmap.N
    )

    # 4. Plotting the Figure
    fig, ax = plt.subplots(
        figsize=(8, 5),
        dpi=300,
    )
    fig.subplots_adjust(
        left=0.1,
        right=0.65,
        top=0.90,
        bottom=0.05
    )

    im = ax.imshow(
        data_masked,
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
    )

    # 5. Legend Configuration
    legend_elements = []
    present_values = np.unique(
        data_masked.compressed()
    )
    # Sort descending: max positive -> 0 -> min negative (Matches Presence Difference)
    present_values_sorted = np.sort(
        present_values
    )[::-1]

    for val in present_values_sorted:
        legend_elements.append(
            Patch(
                facecolor=colors_dict.get(
                    val,
                    "#000"
                ),
                edgecolor="none",
                linewidth=0,
                label=str(
                    val
                ),
            ),
        )

    ax.legend(
        handles=legend_elements,
        title="Change Difference",
        loc="center left",
        bbox_to_anchor=(
            1.02,
            0.5
        ),
        frameon=False,
        fontsize=8,
        title_fontsize=10,
        alignment="left",
        handlelength=2.0,
        handleheight=1.5,
    )

    # 6. Cartographic Elements
    scalebar = ScaleBar(
        pixel_size_m,
        units="m",
        length_fraction=0.35,
        location="lower left",
        box_alpha=0.0
    )
    ax.add_artist(
        scalebar
    )

    north_arrow(
        ax,
        location="upper right",
        shadow=False,
        rotation={
            "degrees": 0
        },
        scale=0.3
    )

    # 7. Axes styling (with Lat/Lon labels)
    ax.set_title(
        "Change Difference",
        fontsize=18,
        pad=5,
    )
    ax.set_aspect(
        "equal"
    )

    to_latlon = Transformer.from_crs(
        src_crs,
        "EPSG:4326",
        always_xy=True,
    )
    height, width = data.shape

    def format_lon(
        x,
        pos
    ):
        x = np.clip(
            x,
            0,
            width - 1
        )
        x_proj, y_proj = rasterio.transform.xy(
            transform,
            height // 2,
            x
        )
        lon, lat = to_latlon.transform(
            x_proj,
            y_proj
        )
        return f"{lon:.1f}°"

    def format_lat(
        y,
        pos
    ):
        y = np.clip(
            y,
            0,
            height - 1
        )
        x_proj, y_proj = rasterio.transform.xy(
            transform,
            y,
            width // 2
        )
        lon, lat = to_latlon.transform(
            x_proj,
            y_proj
        )
        return f"{lat:.1f}°"

    ax.xaxis.set_major_formatter(
        FuncFormatter(
            format_lon
        )
    )
    ax.yaxis.set_major_formatter(
        FuncFormatter(
            format_lat
        )
    )

    ax.xaxis.set_major_locator(
        mticker.MaxNLocator(
            nbins=4
        )
    )
    ax.yaxis.set_major_locator(
        mticker.MaxNLocator(
            nbins=6
        )
    )

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=7,
        pad=4
    )
    plt.setp(
        ax.get_yticklabels(),
        rotation=90,
        va="center"
    )

    maps_dir = os.path.join(
        output_dir,
        "maps"
    )
    os.makedirs(
        maps_dir,
        exist_ok=True
    )

    output_figure_path = os.path.join(
        maps_dir,
        "map_change_difference.png",
    )

    plt.savefig(
        output_figure_path,
        dpi=300,
        format="png",
    )
    plt.show()

    print(
        f"Map successfully saved to: {output_figure_path}"
    )

# Execute the function
plot_change_difference_map(
    output_dir=output_path,
    nodata_val=no_data_value,
    raster_filename="change_map_E_difference.tif",
)


##### 5.2.2.3 Temporal Allocation of Change

In [ ]:
def plot_change_temporal_allocation_map(
    output_dir: str,
    nodata_val: int,
    raster_filename: str = "change_map_F_temporal_allocation.tif",
) -> None:
    """
    Plot the Change Temporal Allocation map using a discrete integer scale.

    0 is represented in gray.
    Positive values use a gradient transitioning from dark red (1)
    to yellow, and finally to light green (maximum value).

    Parameters
    ----------
    output_dir : str
        Directory where the 'rasters' and 'maps' folders are located.
    nodata_val : int
        Pixel value indicating no data.
    raster_filename : str, optional
        Filename of the raster to be plotted. Default is "change_map_F_temporal_allocation.tif".

    Returns
    -------
    None
    """
    # 1. Input Validation and Path Setup
    rasters_dir = os.path.join(
        output_dir,
        "rasters",
    )
    raster_path = os.path.join(
        rasters_dir,
        raster_filename,
    )

    if not os.path.exists(
        raster_path,
    ):
        raise FileNotFoundError(
            f"Raster not found: {raster_path}",
        )

    print(
        f"Reading Change Temporal Allocation map for plotting: {raster_path}",
    )

    pixel_size_m = compute_display_pixel_size_m(
        raster_path=raster_path,
        downsample_divisor=1
    )

    # 2. Data Loading and Masking
    with rasterio.open(
        raster_path,
    ) as src:
        scale_factor = 1
        data = src.read(
            1,
            out_shape=(
                int(
                    src.height * scale_factor
                ),
                int(
                    src.width * scale_factor
                ),
            ),
            resampling=rasterio.enums.Resampling.nearest,
        )

        # Force masking using the provided variable
        data_masked = np.ma.masked_equal(
            data,
            nodata_val,
        )

        left, bottom, right, top = src.bounds
        src_crs = src.crs
        transform = src.transform

    # 3. Determine range and colors
    data_max = int(
        data_masked.max()
    )

    # Ensure safe bounds just in case the map is entirely 0
    if data_max < 1:
        data_max = 1

    colors_dict = {
        0: "#c0c0c0"
    }  # Gray for 0

    # Positive values: Custom gradient Dark Red -> Yellow -> Light Green
    pos_cmap = LinearSegmentedColormap.from_list(
        "red_yellow_green",
        [
            "#8b0000",
            "#ffff00",
            "#90ee90"
        ]
    )

    for i, val in enumerate(
        range(
            1,
            data_max + 1
        )
    ):
        # Normalize fraction between 0 and 1
        if data_max > 1:
            frac = i / (
                data_max - 1
            )
        else:
            frac = 0.5

        colors_dict[val] = pos_cmap(
            frac
        )

    # Build standard lists for colormap
    all_vals = np.arange(
        0,
        data_max + 1
    )
    colors_list = [
        colors_dict.get(
            v,
            "#000000"
        ) for v in all_vals
    ]

    cmap = ListedColormap(
        colors_list
    )

    # Create boundaries for every integer from 0 to max
    bounds = np.arange(
        -0.5,
        data_max + 1.5,
        1
    )
    norm = BoundaryNorm(
        bounds,
        cmap.N
    )

    # 4. Plotting the Figure
    # FIXED FIGURE SIZE AND MARGINS FOR PERFECT OVERLAY
    fig, ax = plt.subplots(
        figsize=(8, 5),
        dpi=300,
    )
    fig.subplots_adjust(
        left=0.1,
        right=0.65,
        top=0.90,
        bottom=0.05
    )

    im = ax.imshow(
        data_masked,
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
    )

    # 5. Legend Configuration
    legend_elements = []
    present_values = np.unique(
        data_masked.compressed()
    )
    # Sort ascending: 0 -> max positive
    present_values_sorted = np.sort(
        present_values
    )

    for val in present_values_sorted:
        legend_elements.append(
            Patch(
                facecolor=colors_dict.get(
                    val,
                    "#000"
                ),
                edgecolor="none",
                linewidth=0,
                label=str(
                    val
                ),
            ),
        )

    ax.legend(
        handles=legend_elements,
        title="Temporal Allocation",
        loc="center left",
        bbox_to_anchor=(
            1.02,
            0.5
        ),
        frameon=False,
        fontsize=8,
        title_fontsize=10,
        alignment="left",
        handlelength=2.0,
        handleheight=1.5,
    )

    # 6. Cartographic Elements
    scalebar = ScaleBar(
        pixel_size_m,
        units="m",
        length_fraction=0.35,
        location="lower left",
        box_alpha=0.0
    )
    ax.add_artist(
        scalebar
    )

    north_arrow(
        ax,
        location="upper right",
        shadow=False,
        rotation={
            "degrees": 0
        },
        scale=0.3
    )

    # 7. Axes styling (with Lat/Lon labels)
    ax.set_title(
        "Change Temporal Allocation",
        fontsize=18,
        pad=5,
    )
    ax.set_aspect(
        "equal"
    )

    to_latlon = Transformer.from_crs(
        src_crs,
        "EPSG:4326",
        always_xy=True,
    )
    height, width = data.shape

    def format_lon(
        x,
        pos
    ):
        x = np.clip(
            x,
            0,
            width - 1
        )
        x_proj, y_proj = rasterio.transform.xy(
            transform,
            height // 2,
            x
        )
        lon, lat = to_latlon.transform(
            x_proj,
            y_proj
        )
        return f"{lon:.1f}°"

    def format_lat(
        y,
        pos
    ):
        y = np.clip(
            y,
            0,
            height - 1
        )
        x_proj, y_proj = rasterio.transform.xy(
            transform,
            y,
            width // 2
        )
        lon, lat = to_latlon.transform(
            x_proj,
            y_proj
        )
        return f"{lat:.1f}°"

    ax.xaxis.set_major_formatter(
        FuncFormatter(
            format_lon
        )
    )
    ax.yaxis.set_major_formatter(
        FuncFormatter(
            format_lat
        )
    )

    ax.xaxis.set_major_locator(
        mticker.MaxNLocator(
            nbins=4
        )
    )
    ax.yaxis.set_major_locator(
        mticker.MaxNLocator(
            nbins=6
        )
    )

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=7,
        pad=4
    )
    plt.setp(
        ax.get_yticklabels(),
        rotation=90,
        va="center"
    )

    maps_dir = os.path.join(
        output_dir,
        "maps"
    )
    os.makedirs(
        maps_dir,
        exist_ok=True
    )

    output_figure_path = os.path.join(
        maps_dir,
        "map_change_temporal_allocation.png",
    )

    # NO BBOX_INCHES TO PRESERVE EXACT FIGURE SIZE
    plt.savefig(
        output_figure_path,
        dpi=300,
        format="png",
    )
    plt.show()

    print(
        f"Map successfully saved to: {output_figure_path}"
    )

# Execute the function
plot_change_temporal_allocation_map(
    output_dir=output_path,
    nodata_val=no_data_value,
    raster_filename="change_map_F_temporal_allocation.tif",
)
